# Zero-shot ASR and fine-tuning pipeline 

In this notebook: <br>

0. Imports & Functions & Global Variables <br>
    0.1. Install and import <br>
    0.2 Custom functions <br>
    0.3 Global variables <br>
1. Environment check
2. Zero-shot ASR with baseline models <br>
    2.1. Multilingual Whisper Large <br>
    2.2. Multilingual Whisper Medium <br>
    2.3. Multilingual Whisper Small <br>
    2.4. BuzzASR/Italian <br>
    2.5. Litus model <br>
    2.6. Sandiago model <br>
3. Preparing Data for fine-tuning <br>
    3.1. Configuring processors <br>
    3.2. Vectorized datasets for Multilingual Whisper and BuzzASR/Italian <br>
    3.3. Vectorized datasets for Litus model <br>
    3.4. Vectorized datasets for Sandiago model <br>
    3.5. Collators <br>
    3.6. Compute metrics <br>
4. Model Fine-tuning <br>
    4.1. Multilingual Whisper <br> 
    4.2. Multilingual Whisper Medium <br>
    4.3. Multilingual Whisper Small <br>
    4.4. BuzzASR/Italian <br>
    4.5. Litus model <br>
    4.6. Sandiago model <br>
5. Predictions on test set after fine-tuning to Romagnol
    5.1. Multilingual Whisper <br> 
    5.2. Multilingual Whisper Medium <br>
    5.3. Multilingual Whisper Small <br>
    5.4. BuzzASR/Italian <br>
    5.5. Litus model <br>
    5.6. Sandiago model <br>
6. ASR evaluation <br>

*N.B. Not ready to run out of the box: adjust local paths, model names, and checkpoint directories before execution.* <br>

N.B.2 Conda and pip dependencies of the GPU environment where this pipeline was developed and run can be found in environment.yml <br>

The models chosen for these ASR experimental setup with Romagnol all rely on seq-to-seq Whisper architecture. In order to assess whether Italian can help with the transcription of Romagnol, Italian-adapted models are deployed alongside generic multilingual models. Their architecture sizes range from small to large, which allows to evaluate scaling capacity. <br>

Six pre-trained models (three generic multilingual and three Italian-adapted) are evaluated out of the box to establish baseline performance. They generic multilingual models are *Whisper Large*, *Whisper Medium*, and *Whisper Small*; the Italian-adapted models are *BuzzASR*, *Litus-ai*, and *Sandiago21*. <br> 

We tested all six models on the test set of two versions of the same dataset: one with a longer average duration per audio clip (*longseg* dataset with an average of 15 s) and one shorter (*shortseg* dataset with an average of 3 seconds). <br>

Models are loaded on Faster Whisper to speed up inference time, and the language tag is set explicitely to Italian. Zero-shot transcriptions are saved for downstream evaluation and comparison with ground-truth references and fine-tuned transcriptions. <br>

Fine-tuning is carried out on all six models using traning and development partitions, separately for *longseg* and *shortseg* datasets. Training is run three times. Once fine-tuning is complete, fine-tuned checkpoints are converted into CTranslate2 format to load them again on Faster Whisper for fast inference. <br>

CER and WER are the evaluation metrics used to evaluate transcriptions in Romagnol. CER is held in higher regard because with its character-level scope it is more indicative of the ability of models to successfully learn and write acoustic signals in Romagnol. WER penalizes entire word-level mismatches, ending up being inflated and not representative of actual performance. <br>


| Phase	| Tool	| Format | Purpose |
| -------- | ------- | -------- | --------- |
| 1. Data Preprocessing & Alignment|	stable-whisper (CTranslate2)|	model.bin (whisper_small_local)	| Ultra-fast inference to chop your audio and text into aligned sentence chunks.
| 2. ASR Fine-Tuning |	Hugging Face / PyTorch (transformers)	| model.safetensors (openai/whisper-...)	| Computes gradients and updates the weigths on the GPU server so the model actually learns Romagnol.


References:

https://openai.com/index/whisper/ <br>
https://huggingface.co/openai/whisper-large-v3 <br>
https://huggingface.co/openai/whisper-medium <br>
https://huggingface.co/openai/whisper-small <br>
https://huggingface.co/BuzzASR/italian?library=transformers <br>
https://huggingface.co/Sandiago21/whisper-large-v2-italian <br>
https://huggingface.co/litus-ai/whisper-small-ita <br>
https://github.com/SYSTRAN/faster-whisper <br>
https://github.com/jianfch/stable-ts <br>
https://towardsai.com/p/machine-learning/whisper-variants-comparison-what-are-their-features-and-how-to-implement-them <br>
https://github.com/openai/whisper <br>
https://whisper-api.com/blog/models/ <br>
https://pypi.org/project/asr_evaluation/ <br>
https://www.gladia.io/blog/fine-tuning-asr-models <br>
https://huggingface.co/vrclc/Malasar_medium_MTF <br>
https://huggingface.co/learn/audio-course/chapter5/fine-tuning <br>
https://nhimg.org/faq/what-is-the-difference-between-character-error-rate-and-word-error-rate-in-speec/#:~:text=The%20practical%20difference%20is%20that%20CER%20can,When%20each%20metric%20is%20the%20better%20fit. <br>
https://huggingface.co/learn/audio-course/chapter5/evaluation <br>
https://github.com/jitsi/jiwer <br>


## 0. Imports & Functions & Global Variables

### 0.1 Install and import

In [1]:
import sys
print("Interpreter:", sys.executable)

Interpreter: /home/ialiani/.conda/envs/asr_env/bin/python


In [ ]:
# N.B. Cell commented out because these libraries were already installed in the Python environment (asr_env) 
# ! pip install asr_evaluation
# ! pip install faster-whisper
# ! pip install tqdm
# ! pip install transformers
# ! pip install ctranslate2
# ! pip install ipywidgets

In [ ]:
from dataclasses import dataclass
import gc
import glob
import json
import os
from pathlib import Path
import random
import shutil
from typing import Any, Dict, List, Optional, Union
import evaluate
import jiwer
import numpy as np
import openpyxl
import pandas as pd
import regex as re
import torch
from ctranslate2.converters import TransformersConverter
from datasets import Audio, Dataset, DatasetDict
from faster_whisper import WhisperModel, download_model
from huggingface_hub import snapshot_download
from tqdm import tqdm as std_tqdm
from transformers import (
    AutoFeatureExtractor,
    EarlyStoppingCallback,
    GenerationConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

/home/ialiani/.conda/envs/asr_env/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


### 0.2 Custom functions

In [ ]:
def env_check():
    """
    Checks environment where Notebook is being run. 
    It uses CUDA if present only if at least 4 GB are free, otherwise it uses CPU. 
    """
    free_vram = torch.cuda.mem_get_info(0)[0] / (1024 ** 3) if torch.cuda.is_available() else 0
    use_gpu = free_vram >= 4.0

    device = "cuda" if use_gpu else "cpu"
    compute_type = "float16" if use_gpu else "int8"

    print(f"Target: {device.upper()} | Compute: {compute_type} (Free VRAM: {free_vram:.1f} GB)")
    return use_gpu, device, compute_type


def setup_asr_model(model_name: str, output_dir: str = None):
    """
    Downloads the model weigths (if not already present) directly from the Hugging Face repository
    as a CTranslate2 binary (float16).
    """
    if output_dir is None:
        output_dir = Path.home() / "thesis_Aliani" / f"{model_name.replace('/', '_')}_weigths"
    output_dir = Path(output_dir)
    if not (output_dir / "model.bin").is_file():
        print(f"Downloading and converting '{model_name}'...")
        local_path = snapshot_download(repo_id=model_name, cache_dir=str(USER_CACHE))
        converter = TransformersConverter(local_path)
        converter.convert(output_dir=str(output_dir), quantization="float16")  
        try:
            AutoFeatureExtractor.from_pretrained(local_path).save_pretrained(str(output_dir))
        except Exception:
            pass
        shutil.rmtree(local_path, ignore_errors=True) # Remove raw snapshot to free disk space
    else:
        print(f"Found existing weigths in: {output_dir}")
    return str(output_dir)


def load_asr_model(model_path, device, compute_type):
    """
    Returns a faster_whisper.WhisperModel instance.
    """
    threads = (os.cpu_count() or 4) if device == "cpu" else 0
    return WhisperModel(
        str(model_path),
        device=device,
        compute_type=compute_type,
        cpu_threads=threads,
        download_root=str(USER_CACHE)  # Keeps internal tiny tokenizer off /models
    )

def transcribe_folder(model, test_partition_dir, output_dir: str):
    """
    Finds all .wav files in test_partition_dir, transcribes them using faster-whisper,
    saves each result as a .txt file in output_dir.
    Accepts str or pathlib.Path for directory arguments.
    """
    test_partition_dir = Path(test_partition_dir) # Convert inputs to Path objects (handles both str and Path instances)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    wav_files = [str(p) for p in test_partition_dir.glob("*.wav")] # Find all .wav files
    print(f"Found {len(wav_files)} .wav files to transcribe.")

    for wav_path in std_tqdm(wav_files, desc="Transcribing audio files", unit="file"):
        filename_no_ext = Path(wav_path).stem
        output_txt_path = output_dir / f"{filename_no_ext}.txt"

        segments, _ = model.transcribe(wav_path, beam_size=5, language="it") # Transcribe audio using faster-whisper
        transcript = " ".join([seg.text.strip() for seg in segments])

        # Save transcription to disk
        with open(output_txt_path, "w", encoding="utf-8") as f:
            f.write(transcript)

    print("\nTranscription complete!")


def load_split_to_dict(split_dir: Path):
    """
    All .txt and .wav are paired in a dictionary.
    """
    audio_paths = []
    transcripts = []

    for wav_path in sorted(split_dir.glob("*.wav")):
        txt_path = wav_path.with_suffix(".txt")
        if txt_path.exists():
            with open(txt_path, "r", encoding="utf-8") as f:
                text = f.read().strip()
            if text:
                audio_paths.append(str(wav_path))
                transcripts.append(text)

    return {"audio": audio_paths, "sentence": transcripts}


def prepare_dataset(batch, processor):
    """
    Converts raw audio to log-Mel spectrograms and tokenizes transcripts into label IDs.
    """
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], 
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Pads variable-length audio features and text labels into a uniform batch tensor.
    """
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        """
        Pads inputs and labels, masks padding tokens with -100, and strips leading Beginnin of Sequence tokens.
        """
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        batch["input_features"] = batch["input_features"].to(torch.float32) # Audio features (remain float32)

        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt") # Text labels
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        batch["decoder_attention_mask"] = labels_batch["attention_mask"]

        return batch

def build_compute_metrics(tokenizer):
    """
    Returns a Trainer metric callback configured with the given tokenizer.
    """
    def compute_metrics(pred):
        """
        Decodes model predictions and computes WER and CER percentages.
        """
        pred_ids = pred.predictions
        label_ids = pred.label_ids

        # Unmask padding tokens
        label_ids[label_ids == -100] = tokenizer.pad_token_id

        # Decode tokens to strings
        pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

        # Compute both metrics (scaled to 0-100%)
        wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
        cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

        return {"wer": wer, "cer": cer}
    
    return compute_metrics


def get_training_args(model_name: str, is_small_model: bool = False):
    """
    Builds shortseg and longseg Seq2SeqTrainingArguments scaled to model size.
    """
    per_device_train = 8 if is_small_model else 2
    grad_accum = 4 if is_small_model else 16
    per_device_eval = 8 if is_small_model else 2

    # Use slightly higher LR for small models (e.g. 5e-5), but conservative LR for large models
    lr = 5e-5 if is_small_model else 2.5e-6

    def make_args(dataset_type: str):
        path = Path.home() / "thesis_Aliani" / "checkpoints" / model_name / dataset_type
        path.mkdir(parents=True, exist_ok=True)
        return Seq2SeqTrainingArguments(
            output_dir=str(path),
            per_device_train_batch_size=per_device_train,
            gradient_accumulation_steps=grad_accum,
            per_device_eval_batch_size=per_device_eval,
            learning_rate=lr,
            weight_decay=0.01,
            lr_scheduler_type="linear",
            warmup_steps=150 if not is_small_model else 300,
            max_steps=1000 if not is_small_model else 2000,
            optim="adamw_bnb_8bit",
            seed=999,
            data_seed=999,
            fp16=True,
            gradient_checkpointing=True,
            predict_with_generate=True,
            generation_max_length=225,
            eval_strategy="steps",
            eval_steps=100,
            save_steps=100,
            save_total_limit=2,
            logging_steps=25,
            report_to=["tensorboard"],
            metric_for_best_model="cer",
            greater_is_better=False,
            load_best_model_at_end=True,
        )

    return {
        "shortseg": make_args("shortseg"),
        "longseg": make_args("longseg"),
    }


def get_normalizer(remove_punctuation: bool = False) -> jiwer.Compose:
    """
    Builds a text normalization transformation.
    Preserves diacritics and accents by default.
    """
    steps = [
        jiwer.ToLowerCase(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
    ]
    if remove_punctuation:
        steps.insert(1, jiwer.RemovePunctuation())
    return jiwer.Compose(steps)


def evaluate_predictions(references: List[str], hypotheses: List[str], apply_norm: bool = True, remove_punctuation: bool = False) -> Dict[str, float]:
    """
    Computes aggregate WER and CER using jiwer.
    Automatically filters out samples with empty reference strings.
    """
    if len(references) != len(hypotheses):
        raise ValueError(
            f"Length mismatch: {len(references)} references vs {len(hypotheses)} hypotheses."
        )

    if apply_norm:
        normalizer = get_normalizer(remove_punctuation=remove_punctuation)
        refs_proc = [normalizer(str(r)) for r in references]
        hyps_proc = [normalizer(str(h)) for h in hypotheses]
    else:
        refs_proc = [str(r).strip() for r in references]
        hyps_proc = [str(h).strip() for h in hypotheses]

    clean_refs, clean_hyps = [], []
    for r, h in zip(refs_proc, hyps_proc):
        if len(r) > 0:
            clean_refs.append(r)
            clean_hyps.append(h)

    if not clean_refs:
        raise ValueError("All references are empty after normalization.")

    wer_score = jiwer.wer(clean_refs, clean_hyps)
    cer_score = jiwer.cer(clean_refs, clean_hyps)

    return {
        "wer": wer_score * 100.0,
        "cer": cer_score * 100.0,
        "samples": len(clean_refs),
    }


def compare_asr_runs(references: List[str], baseline_predictions: List[str], finetuned_predictions: List[str], apply_norm: bool = True, remove_punctuation: bool = False) -> pd.DataFrame:
    """
    Aggregates together the WER and CER scores obtained by all models out of the box and after fine-tuning.
    """
    base_res = evaluate_predictions(
        references,
        baseline_predictions,
        apply_norm=apply_norm,
        remove_punctuation=remove_punctuation,
    )
    ft_res = evaluate_predictions(
        references,
        finetuned_predictions,
        apply_norm=apply_norm,
        remove_punctuation=remove_punctuation,
    )

    df_results = pd.DataFrame([
        {
            "Metric": "WER (%)",
            "Baseline": base_res["wer"],
            "Fine-Tuned": ft_res["wer"],
            "Delta": ft_res["wer"] - base_res["wer"],
        },
        {
            "Metric": "CER (%)",
            "Baseline": base_res["cer"],
            "Fine-Tuned": ft_res["cer"],
            "Delta": ft_res["cer"] - base_res["cer"],
        },
    ]).round(2)

    return df_results

norm = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip()
])


def evaluate_and_append_html(gt_dir, base_dir, ft_dir, model_name, partition, output_html="asr_results.html"):
    """
    Evaluates zero-shot and fine-tuned predictions, comparing them to the ground-truth references in Romagnol.
    """
    gt_dir, base_dir, ft_dir = Path(gt_dir), Path(base_dir), Path(ft_dir)
    out_html = Path(output_html)
    out_csv = out_html.with_suffix(".csv")

    # Read and normalize matching .txt files
    files = sorted([f.name for f in gt_dir.glob("*.txt")])
    refs  = [norm((gt_dir / f).read_text(encoding="utf-8")) for f in files]
    base  = [norm((base_dir / f).read_text(encoding="utf-8")) for f in files]
    ft    = [norm((ft_dir / f).read_text(encoding="utf-8")) for f in files]

    # Filter out any empty ground-truth references
    valid = [(r, b, f) for r, b, f in zip(refs, base, ft) if r]
    refs, base, ft = zip(*valid)

    # Compute aggregate metrics
    b_wer = round(jiwer.wer(list(refs), list(base)) * 100, 2)
    b_cer = round(jiwer.cer(list(refs), list(base)) * 100, 2)
    f_wer = round(jiwer.wer(list(refs), list(ft)) * 100, 2)
    f_cer = round(jiwer.cer(list(refs), list(ft)) * 100, 2)

    df_new = pd.DataFrame([
        {"Model": model_name, "Partition": partition, "Stage": "Baseline",   "Samples": len(refs), "WER (%)": b_wer, "CER (%)": b_cer, "Δ WER": "-", "Δ CER": "-"},
        {"Model": model_name, "Partition": partition, "Stage": "Fine-Tuned", "Samples": len(refs), "WER (%)": f_wer, "CER (%)": f_cer, "Δ WER": f"{f_wer - b_wer:+.2f}", "Δ CER": f"{f_cer - b_cer:+.2f}"}
    ], columns=COLUMNS)

    if out_csv.exists():    # Load existing CSV if schema matches, otherwise start fresh
        try:
            df_old = pd.read_csv(out_csv)
            if list(df_old.columns) == COLUMNS:
                # Remove prior entries for this exact Model + Partition to avoid duplicate rows
                mask = ~((df_old["Model"] == model_name) & (df_old["Partition"] == partition))
                df_all = pd.concat([df_old[mask], df_new], ignore_index=True)
            else:
                df_all = df_new
        except Exception:
            df_all = df_new
    else:
        df_all = df_new

    df_all.to_csv(out_csv, index=False) # Save CSV
    html_table = df_all.to_html(index=False, border=1, justify="center")    # Convert to HTML to save it
    out_html.write_text(
        f"<html><head><style>table {{border-collapse: collapse; font-family: sans-serif;}} th, td {{padding: 8px 14px; text-align: center;}} tr:nth-child(even) {{background: #f9f9f9;}}</style></head><body>{html_table}</body></html>",
        encoding="utf-8"
    )

    print(f"[{model_name} | {partition}]")
    print(f"  WER -> Base: {b_wer}% | FT: {f_wer}% (Δ: {f_wer - b_wer:+.2f}%)")
    print(f"  CER -> Base: {b_cer}% | FT: {f_cer}% (Δ: {f_cer - b_cer:+.2f}%)")


def extract_story_id(stem: str) -> str:
    """
    Strips trailing segment/chunk identifiers to get the base story ID.
    """
    return re.sub(r'_(seg|chunk)\d+$', '', stem)

def export_story_inspection(gt_dirs: dict, model_dirs, num_stories: int = 50, seed: int = 42, output_prefix: str = "inspected_50_stories"):
    """
    Extracts zero-shot and fine-tuned transcriptions for 50 random stories together with their ground-truth references.
    Returns a pandas DataFrame ready for inspection.
    """
    if isinstance(model_dirs, (list, tuple)):
        model_dirs = {"shortseg": model_dirs[0], "longseg": model_dirs[1]}

    gt_short = Path(gt_dirs["shortseg"])
    gt_long = Path(gt_dirs["longseg"])

    # Map each story ID to its exact filenames in shortseg (e.g. ['..._seg001', '..._seg002'])
    short_story_files = {}
    for f in sorted(gt_short.glob("*.txt")):
        s_id = extract_story_id(f.stem)
        short_story_files.setdefault(s_id, []).append(f.name)

    # Map each story ID to its exact filenames in longseg (e.g. ['..._chunk001', '..._chunk002'])
    long_story_files = {}
    for f in sorted(gt_long.glob("*.txt")):
        s_id = extract_story_id(f.stem)
        long_story_files.setdefault(s_id, []).append(f.name)

    # Find stories common to both partitions and sample deterministically
    common_stories = sorted(set(short_story_files.keys()) & set(long_story_files.keys()))
    print(f"Total stories in Shortseg: {len(short_story_files)} | Longseg: {len(long_story_files)}")
    print(f"Common stories found across both partitions: {len(common_stories)}")

    if not common_stories:
        raise ValueError("No common stories found! Check your ground truth paths and regex.")

    sample_k = min(num_stories, len(common_stories))
    selected_stories = sorted(random.Random(seed).sample(common_stories, sample_k))
    print(f"Extracting all segments/chunks for {sample_k} sampled stories...")

    rows = []
    for idx, story_id in enumerate(selected_stories, 1):
        # Process shortseg first, then longseg for this exact story
        for partition, gt_dir, files_map, models in [
            ("shortseg", gt_short, short_story_files, model_dirs["shortseg"]),
            ("longseg",  gt_long,  long_story_files,  model_dirs["longseg"])
        ]:
            # Exact filenames found in the ground truth for this story & partition
            file_names = sorted(files_map[story_id])

            for fname in file_names:
                ref_file = gt_dir / fname
                if not ref_file.exists():
                    continue

                ref_raw = ref_file.read_text(encoding="utf-8").strip()
                ref_clean = norm(ref_raw)

                # Ground Truth row
                rows.append({
                    "Story": f"Story {idx:02d}",
                    "Story ID": story_id,
                    "Partition": partition,
                    "File": fname,
                    "System": "GROUND TRUTH",
                    "Transcript": ref_raw,
                    "WER (%)": "-",
                    "CER (%)": "-"
                })

                # Check all models for this exact filename in their respective directory
                for model_name, (base_dir, ft_dir) in models.items():
                    for stage, path in [("Base", Path(base_dir)), ("Fine-Tuned", Path(ft_dir))]:
                        pred_file = path / fname
                        hyp_raw = pred_file.read_text(encoding="utf-8").strip() if pred_file.exists() else ""
                        hyp_clean = norm(hyp_raw)

                        if ref_clean and hyp_clean:
                            wer = round(jiwer.wer(ref_clean, hyp_clean) * 100.0, 2)
                            cer = round(jiwer.cer(ref_clean, hyp_clean) * 100.0, 2)
                        elif ref_clean and not hyp_clean:
                            wer, cer = 100.0, 100.0
                        else:
                            wer, cer = 0.0, 0.0

                        rows.append({
                            "Story": f"Story {idx:02d}",
                            "Story ID": story_id,
                            "Partition": partition,
                            "File": fname,
                            "System": f"{model_name} ({stage})",
                            "Transcript": hyp_raw,
                            "WER (%)": wer,
                            "CER (%)": cer
                        })

                # Blank spacer between audio pieces (seg or chunk)
                rows.append({col: "" for col in rows[0]})

        # Visual boundary between stories
        rows.append({col: "---" for col in rows[0]})

    # Save clean DataFrame to Excel and HTML
    df = pd.DataFrame(rows)
    df.to_excel(f"{output_prefix}.xlsx", index=False)
    
    html_content = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; padding: 20px; }}
  table {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
  th, td {{ border: 1px solid #ddd; padding: 6px 10px; }}
  th {{ background-color: #2c3e50; color: white; position: sticky; top: 0; }}
  tr:has(td:contains('GROUND TRUTH')) {{ background-color: #eaf2f8; font-weight: 600; }}
  tr:has(td:contains('(Fine-Tuned)')) {{ background-color: #fef9e7; }}
  tr:has(td:contains('---')) {{ background-color: #34495e; height: 10px; }}
</style>
</head>
<body>
  <h2>Qualitative Story Inspection (Shortseg + Longseg)</h2>
  {df.to_html(index=False, border=0)}
</body>
</html>"""
    Path(f"{output_prefix}.html").write_text(html_content, encoding="utf-8")
    print(f"Successfully generated {output_prefix}.xlsx and {output_prefix}.html")



def aggregate_runs_and_export(csv_files: list[str | Path], output_prefix: str = "asr_aggregated_results"):
    """
    Loads evaluation CSV files from independent runs, calculates mean and std
    for WER, CER, Δ WER, and Δ CER across runs, safely handling models with
    fewer runs (i.e. BuzzASR), and exports to CSV, HTML, and LaTeX.
    """
    dfs = [Path(f) if isinstance(f, str) else f for f in csv_files]
    loaded_dfs = []

    for f in dfs:
        if not f.exists():
            print(f"[Warning] Run CSV file not found: {f}. Skipping.")
            continue
        df = pd.read_csv(f)
        # Convert delta columns to numeric float (stripping '-' for baselines)
        df["Δ WER_num"] = pd.to_numeric(df["Δ WER"], errors="coerce")
        df["Δ CER_num"] = pd.to_numeric(df["Δ CER"], errors="coerce")
        loaded_dfs.append(df)

    if not loaded_dfs:
        raise ValueError("No valid CSV files were loaded.")

    combined_df = pd.concat(loaded_dfs, ignore_index=True)  # Concatenate all runs together

    group_cols = ["Model", "Partition", "Stage"]

    def get_stats(series):
        """ 
        Handles BuzzASR single-run case cleanly
        """
        valid_series = series.dropna()
        if len(valid_series) == 0:
            return pd.Series({"mean": np.nan, "std": 0.0, "count": 0})
        elif len(valid_series) == 1:
            return pd.Series(
                {"mean": valid_series.iloc[0], "std": 0.0, "count": 1}
            )
        else:
            return pd.Series(
                {
                    "mean": valid_series.mean(),
                    "std": valid_series.std(ddof=1),
                    "count": len(valid_series),
                }
            )

    rows = []
    
    for (model, partition, stage), group in combined_df.groupby(    # Group by Model, Partition, and Stage
        group_cols, sort=False
    ):
        samples = group["Samples"].iloc[0]

        wer_stats = get_stats(group["WER (%)"])
        cer_stats = get_stats(group["CER (%)"])
        d_wer_stats = get_stats(group["Δ WER_num"])
        d_cer_stats = get_stats(group["Δ CER_num"])

        n_runs = int(wer_stats["count"])
        is_baseline = stage == "Baseline"

        # Format values: if only 1 run, display metric directly without ± std (or ± 0.00)
        if n_runs == 1:
            wer_str = f"{wer_stats['mean']:.2f}"
            cer_str = f"{cer_stats['mean']:.2f}"
            d_wer_str = (
                "-" if is_baseline else f"{d_wer_stats['mean']:+.2f} (1 run)"
            )
            d_cer_str = (
                "-" if is_baseline else f"{d_cer_stats['mean']:+.2f} (1 run)"
            )
        else:
            wer_str = f"{wer_stats['mean']:.2f} ± {wer_stats['std']:.2f}"
            cer_str = f"{cer_stats['mean']:.2f} ± {cer_stats['std']:.2f}"
            if is_baseline:
                d_wer_str = "-"
                d_cer_str = "-"
            else:
                d_wer_str = f"{d_wer_stats['mean']:+.2f} ± {d_wer_stats['std']:.2f}"
                d_cer_str = f"{d_cer_stats['mean']:+.2f} ± {d_cer_stats['std']:.2f}"

        rows.append(
            {
                "Model": model,
                "Partition": partition,
                "Stage": stage,
                "Runs": n_runs,
                "Samples": samples,
                "WER (%)": wer_str,
                "CER (%)": cer_str,
                "Δ WER (%)": d_wer_str,
                "Δ CER (%)": d_cer_str,
            }
        )

    final_df = pd.DataFrame(rows)
    
    csv_out = Path(f"{output_prefix}.csv")  # Export Clean CSV
    final_df.to_csv(csv_out, index=False)

    html_out = Path(f"{output_prefix}.html")  # Export HTML Report
    html_table = final_df.to_html(index=False, border=1, justify="center")
    html_content = (
        "<html><head><style>"
        "table {border-collapse: collapse; font-family: sans-serif; font-size: 14px; margin: auto;}"
        "th, td {padding: 8px 14px; text-align: center; border: 1px solid #ddd;}"
        "th {background-color: #f2f2f2; font-weight: bold;}"
        "tr:nth-child(even) {background-color: #f9f9f9;}"
        "h2 {text-align: center; font-family: sans-serif;}"
        "</style></head><body>"
        f"<h2>ASR Evaluation: Aggregated Results across Runs (Mean ± Std)</h2>"
        f"{html_table}</body></html>"
    )
    html_out.write_text(html_content, encoding="utf-8")
    latex_out = Path(f"{output_prefix}.tex")    # Export Booktabs LaTeX Table for Thesis
    latex_table = final_df.to_latex(
        index=False,
        caption="ASR performance metrics across independent training runs reported as mean $\\pm$ standard deviation.",
        label="tab:asr_aggregated_results",
        escape=False,
    )
    latex_out.write_text(latex_table, encoding="utf-8")

    print(f"Aggregated results generated successfully:")
    print(f"  - CSV:   {csv_out.resolve()}")
    print(f"  - HTML:  {html_out.resolve()}")
    print(f"  - LaTeX: {latex_out.resolve()}")

    return final_df

### 0.3 Global variables

In [3]:
# Run this cell if you need to run the notebook locally.
# To (un)comment out all lines: CTRL + A (select everything in this cell) > CTRL + ù (uncomment every line)
 
# OUTPUT_DIR = Path.home() / "OneDrive - Alma Mater Studiorum Università di Bologna" / "Master's Dissertation" / "Pre-processing_output"
# DATASET_SPLIT = OUTPUT_DIR / "dataset_split"
# DATASET_SPLIT.mkdir(parents=True, exist_ok=True)

# train_set = DATASET_SPLIT / "train"
# dev_set = DATASET_SPLIT / "dev"
# test_set = DATASET_SPLIT / "test"

# ZERO_SHOT = DATASET_SPLIT / "zero_shot"
# ZERO_SHOT.mkdir(parents=True, exist_ok=True)

# MULTI_ZERO_SHOT = ZERO_SHOT / "multi_zero_shot"
# MULTI_ZERO_SHOT.mkdir(parents=True, exist_ok=True)

# ITA_ZERO_SHOT = ZERO_SHOT / "ita_zero_shot"
# ITA_ZERO_SHOT.mkdir(parents=True, exist_ok=True)

# ITA_BUZZ = ITA_ZERO_SHOT / "ita_buzz"
# ITA_BUZZ.mkdir(parents=True, exist_ok=True)

# ITA_LITUS = ITA_ZERO_SHOT / "ita_litus"
# ITA_LITUS.mkdir(parents=True, exist_ok=True)

# TEST_AUDIO_1 = OUTPUT_DIR / "dataset_split" / "test" / "rgn_bar_1_st1_seg001.wav"
# TEST_TEXT_1 = OUTPUT_DIR / "dataset_split" / "test" / "rgn_bar_1_st1_seg001.txt"

# TEST_AUDIO_2 = OUTPUT_DIR / "dataset_split" / "test" / "rgn_lug_5_seg001.wav"
# TEST_TEXT_2 = OUTPUT_DIR / "dataset_split" / "test" / "rgn_lug_5_seg001.txt"

In [ ]:
BASE_DIR = Path.home() / "thesis_Aliani" # Base project folder on the server

# Set cache directories to your personal user storage
USER_CACHE = Path.home() / "thesis_Aliani" / ".cache" / "huggingface"
USER_CACHE.mkdir(parents=True, exist_ok=True)

# Paths to two datasets with Romagnol data
SHORTSEG_DATASET = BASE_DIR / "shortseg_dataset_split"
LONGSEG_DATASET = BASE_DIR / "longseg_dataset_split"

# Shortseg dataset partitions 
DEV_DIR_SHORTSEG = SHORTSEG_DATASET / "dev"
TEST_DIR_SHORTSEG = SHORTSEG_DATASET / "test"
TRAIN_DIR_SHORTSEG = SHORTSEG_DATASET / "train"

# Longseg dataset partitions
DEV_DIR_LONGSEG = LONGSEG_DATASET / "dev"
TEST_DIR_LONGSEG = LONGSEG_DATASET / "test"
TRAIN_DIR_LONGSEG = LONGSEG_DATASET / "train"

BASE_MODELS_DIR = BASE_DIR / "baseline_models"

# Output directories for model predictions & evaluation metrics
PREDICTIONS_DIR = BASE_DIR / "predictions"

BASELINE_RES_DIR = PREDICTIONS_DIR/ "baseline_results"
BASELINE_RES_DIR_SHORTSEG = BASELINE_RES_DIR / "baseline_shortseg"
BASELINE_RES_DIR_LONGSEG = BASELINE_RES_DIR / "baseline_longseg"

FINETUNED_RES_DIR = PREDICTIONS_DIR / "fine-tuned_results_3"
FINETUNED_RES_DIR_SHORTSEG = FINETUNED_RES_DIR / "finetuned_shortseg_3"
FINETUNED_RES_DIR_LONGSEG = FINETUNED_RES_DIR / "finetuned_longseg_3"

# results directories for shortseg and longseg dataset for baseline models
MULTI_WHISP_L_SHORTSEG = BASELINE_RES_DIR_SHORTSEG / "multi_whisper_l"
MULTI_WHISP_M_SHORTSEG = BASELINE_RES_DIR_SHORTSEG / "multi_whisper_m"
MULTI_WHISP_S_SHORTSEG = BASELINE_RES_DIR_SHORTSEG / "multi_whisper_s"
ITA_BUZZ_SHORTSEG = BASELINE_RES_DIR_SHORTSEG / "ita_buzz"
ITA_LITUS_SHORTSEG = BASELINE_RES_DIR_SHORTSEG / "ita_litus"
ITA_SANDIAGO_SHORTSEG = BASELINE_RES_DIR_SHORTSEG / "ita_sandiago"

MULTI_WHISP_L_LONGSEG = BASELINE_RES_DIR_LONGSEG / "multi_whisper_l"
MULTI_WHISP_M_LONGSEG = BASELINE_RES_DIR_LONGSEG / "multi_whisper_m"
MULTI_WHISP_S_LONGSEG = BASELINE_RES_DIR_LONGSEG / "multi_whisper_s"
ITA_BUZZ_LONGSEG = BASELINE_RES_DIR_LONGSEG / "ita_buzz"
ITA_LITUS_LONGSEG = BASELINE_RES_DIR_LONGSEG / "ita_litus"
ITA_SANDIAGO_LONGSEG = BASELINE_RES_DIR_LONGSEG / "ita_sandiago"

# results directories for shortseg and longseg dataset for fine-tuned models
MULTI_WHISP_L_SHORTSEG_RGN = FINETUNED_RES_DIR_SHORTSEG / "multi_whisper_l"
MULTI_WHISP_M_SHORTSEG_RGN = FINETUNED_RES_DIR_SHORTSEG / "multi_whisper_m"
MULTI_WHISP_S_SHORTSEG_RGN = FINETUNED_RES_DIR_SHORTSEG / "multi_whisper_s"
ITA_BUZZ_SHORTSEG_RGN = FINETUNED_RES_DIR_SHORTSEG / "ita_buzz"
ITA_LITUS_SHORTSEG_RGN = FINETUNED_RES_DIR_SHORTSEG / "ita_litus"
ITA_SANDIAGO_SHORTSEG_RGN = FINETUNED_RES_DIR_SHORTSEG / "ita_sandiago"

MULTI_WHISP_L_LONGSEG_RGN = FINETUNED_RES_DIR_LONGSEG / "multi_whisper_l"
MULTI_WHISP_M_LONGSEG_RGN = FINETUNED_RES_DIR_LONGSEG / "multi_whisper_m"
MULTI_WHISP_S_LONGSEG_RGN = FINETUNED_RES_DIR_LONGSEG / "multi_whisper_s"
ITA_BUZZ_LONGSEG_RGN = FINETUNED_RES_DIR_LONGSEG / "ita_buzz"
ITA_LITUS_LONGSEG_RGN = FINETUNED_RES_DIR_LONGSEG / "ita_litus"
ITA_SANDIAGO_LONGSEG_RGN = FINETUNED_RES_DIR_LONGSEG / "ita_sandiago"

COLUMNS = ["Model", "Partition", "Stage", "Samples", "WER (%)", "CER (%)", "Δ WER", "Δ CER"]

# Model mappings for shortseg
results_shortseg = {
    "Multi Whisper Large": (MULTI_WHISP_L_SHORTSEG, MULTI_WHISP_L_SHORTSEG_RGN),
    "Multi Whisper Medium" : (MULTI_WHISP_M_SHORTSEG, MULTI_WHISP_M_SHORTSEG_RGN),
    "Multi Whisper Small" : (MULTI_WHISP_S_SHORTSEG, MULTI_WHISP_S_SHORTSEG_RGN),
    "ITA Buzz":      (ITA_BUZZ_SHORTSEG,    ITA_BUZZ_SHORTSEG_RGN),
    "ITA Litus":     (ITA_LITUS_SHORTSEG,   ITA_LITUS_SHORTSEG_RGN),
    "ITA Sandiago":  (ITA_SANDIAGO_SHORTSEG,ITA_SANDIAGO_SHORTSEG_RGN),
}

# Model mappings for longseg
results_longseg = {
    "Multi Whisper Large": (MULTI_WHISP_L_LONGSEG, MULTI_WHISP_L_LONGSEG_RGN),
    "Multi Whisper Medium" : (MULTI_WHISP_M_LONGSEG, MULTI_WHISP_M_LONGSEG_RGN),
    "Multi Whisper Small" : (MULTI_WHISP_S_LONGSEG, MULTI_WHISP_S_LONGSEG_RGN),
    "ITA Buzz":      (ITA_BUZZ_LONGSEG,    ITA_BUZZ_LONGSEG_RGN),
    "ITA Litus":     (ITA_LITUS_LONGSEG,   ITA_LITUS_LONGSEG_RGN),
    "ITA Sandiago":  (ITA_SANDIAGO_LONGSEG,ITA_SANDIAGO_LONGSEG_RGN),
}

TEST_AUDIO_1 = TEST_DIR_LONGSEG / "rgn_guer_1_st6_chunk003.wav"
TEST_AUDIO_2 = TEST_DIR_LONGSEG / "rgn_bell_2_st1_chunk001.wav"
TEST_AUDIO_3 = TEST_DIR_SHORTSEG / "rgn_bar_1_st1_seg001.wav"
TEST_AUDIO_4 = TEST_DIR_SHORTSEG / "rgn_gal_9_seg017.wav"
test_audios = [TEST_AUDIO_1, TEST_AUDIO_2, TEST_AUDIO_3, TEST_AUDIO_4]

# Define clean destination path for multi_whisp shortseg
FINAL_MODELS_DIR = BASE_DIR / "final_models"

# # Automatically create the output folders if they do not exist
# # for out_path in [
#     SHORTSEG_DATASET, LONGSEG_DATASET, PREDICTIONS_DIR,
#     BASE_MODELS_DIR, BASELINE_RES_DIR, BASELINE_RES_DIR_SHORTSEG, BASELINE_RES_DIR_LONGSEG, 
#     MULTI_WHISP_L_SHORTSEG, MULTI_WHISP_M_SHORTSEG, MULTI_WHISP_S_SHORTSEG, ITA_BUZZ_SHORTSEG, ITA_LITUS_SHORTSEG, ITA_SANDIAGO_SHORTSEG,
#     MULTI_WHISP_L_LONGSEG, MULTI_WHISP_M_LONGSEG, MULTI_WHISP_S_LONGSEG, ITA_BUZZ_LONGSEG, ITA_LITUS_LONGSEG, ITA_SANDIAGO_LONGSEG, 
#     MULTI_WHISP_L_SHORTSEG_RGN, MULTI_WHISP_M_SHORTSEG_RGN, MULTI_WHISP_S_SHORTSEG_RGN, ITA_BUZZ_SHORTSEG_RGN, ITA_LITUS_SHORTSEG_RGN, ITA_SANDIAGO_SHORTSEG_RGN,
#     MULTI_WHISP_L_LONGSEG_RGN, MULTI_WHISP_M_LONGSEG_RGN, MULTI_WHISP_S_LONGSEG_RGN, ITA_BUZZ_LONGSEG_RGN, ITA_LITUS_LONGSEG_RGN, ITA_SANDIAGO_LONGSEG_RGN,
#     FINAL_MODELS_DIR,
# ]:
#     out_path.mkdir(parents=True, exist_ok=True)

## 1. Environment check
N.B. The downloaded and converted weigths on disk are identical regardless of whether you run on CPU or GPU. <br>
The difference between CPU and GPU execution happens strictly inside load_asr_model during runtime:
> * When targeting GPU: <br>
device="cuda" and compute_type="float16" load the float16 weigths straight into GPU VRAM for native tensor core math.   
> * When targeting CPU: <br>
device="cpu" and compute_type="int8" instruct CTranslate2 to dynamically quantize or map the operations into 8-bit integers in system RAM on the fly. CTranslate2 handles this conversion automatically at load time.

In [5]:
is_cuda, device, compute_type = env_check()

Target: CUDA | Compute: float16 (Free VRAM: 31.2 GB)


## 2. Zero-shot ASR with baseline models
What happens behind the scenes: downloading weigths, loading them locally via CTranslate2 / faster-whisper and running the models locally (both on laptop and on the server) <br>
* setup_asr_model selects the most suitable size for the specified model based on the the environment the code is being run on and downloads them. It the weigths for as ASR model already exist inside the current working directory, it skips downloading and simply returns the absolute path string.
* load_asr_model takes that path string and initializes faster-whisper.
* transcribe_folder processes all .wav files from your input folder and outputs .txt transcripts to your destination folder.

### 2.1 Multilingual Whisper Large

In [ ]:
# Setup/convert Whisper Large
model_dir = setup_asr_model(
    model_name="openai/whisper-large-v3",
    output_dir=BASE_MODELS_DIR / "openai_whisper-large_weigths"
)

# Load model
multi_whisp_l_model = load_asr_model(
    model_path=model_dir,
    device=device,
    compute_type=compute_type
)

In [ ]:
# Transcription test with single audio files
for i, audio in enumerate(test_audios, 1):
    segments, info = multi_whisp_l_model.transcribe(audio, beam_size=5, language="it")
    transcription = " ".join(seg.text for seg in segments)
    print(f"TEST_AUDIO_{i}: {transcription}\n")

In [ ]:
# N.B. Cell commented out so as not to run it by accident.
# Run this cell to transcribe test set of shortseg dataset.
# transcribe_folder(model=multi_whisp_l_model, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=MULTI_WHISP_L_SHORTSEG)

In [ ]:
# N.B. Cell commented out so as not to run it by accident.
# Run this cell to transcribe test set of longseg dataset. 
# transcribe_folder(model=multi_whisp_l_model, test_partition_dir=TEST_DIR_LONGSEG, output_dir=MULTI_WHISP_L_LONGSEG)

In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA.
del multi_whisp_l_model
 
gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

### 2.2 Whisper medium

In [ ]:
# Setup/convert Whisper Medium
model_dir = setup_asr_model(
    model_name="openai/whisper-medium",
    output_dir=BASE_MODELS_DIR / "openai_whisper-medium_weigths"
)

# Load model
multi_whisp_m_model = load_asr_model(
    model_path=model_dir,
    device=device,
    compute_type=compute_type
)

Found existing weigths in: /home/ialiani/thesis_Aliani/baseline_models/openai_whisper-medium_weigths


In [6]:
# Transcription test with single audio files
for i, audio in enumerate(test_audios, 1):
    segments, info = multi_whisp_m_model.transcribe(audio, beam_size=5, language="it")
    transcription = " ".join(seg.text for seg in segments)
    print(f"TEST_AUDIO_{i}: {transcription}\n")

TEST_AUDIO_1:  E lì, la ciamelana mazzumere.  Ui! Dove siete?  Sono qui in terra.  Stai bene?  Che mi dici? Me che con te?

TEST_AUDIO_2:  Nocca spirda in di Taip, brisa da Deas.  Driaglior cagliada veni o i decendi.  Un sginleo nerapeas.  Uia di post, esuia di pinxir,  in docchè l'om de sil, e pe alora t'lanot.

TEST_AUDIO_3:  evento a Tramontena e Aesol.

TEST_AUDIO_4:  Ad escloriv, idet la Christi.



In [ ]:
# N.B. Cell commented out so as not to run it by accident.
# Run this cell to transcribe test set of shortseg dataset.
# transcribe_folder(model=multi_whisp_m_model, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=MULTI_WHISP_M_SHORTSEG)

Found 875 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 875/875 [05:36<00:00,  2.60file/s]


Transcription complete!


In [ ]:
# N.B. Cell commented out so as not to run it by accident.
# Run this cell to transcribe test set of longseg dataset. 
# transcribe_folder(model=multi_whisp_m_model, test_partition_dir=TEST_DIR_LONGSEG, output_dir=MULTI_WHISP_M_LONGSEG)

Found 206 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 206/206 [02:08<00:00,  1.61file/s]


Transcription complete!


In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del multi_whisp_m_model

gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

### 2.3 Whisper small

In [ ]:
# Setup/convert Whisper Small
model_dir = setup_asr_model(
    model_name="openai/whisper-small",
    output_dir=BASE_MODELS_DIR / "openai_whisper-small_weigths"
)

# Load model
multi_whisp_s_model = load_asr_model(
    model_path=model_dir,
    device=device,
    compute_type=compute_type
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [11]:
# Transcription test with single audio files
for i, audio in enumerate(test_audios, 1):
    segments, info = multi_whisp_s_model.transcribe(audio, beam_size=5, language="it")
    transcription = " ".join(seg.text for seg in segments)
    print(f"TEST_AUDIO_{i}: {transcription}\n")

TEST_AUDIO_1:  E lì, la ciamelone mazzumare.  Ui, dove siete?  Sono i quai ne teren.  Stai bene!  Che che mi dì, mei che con te!

TEST_AUDIO_2:  No, che spirdain di type, brisa da dech,  Dei lior caglià da veni oi decendee,  Uns guinele a un arapes,  Qui è di post, e suia di pinchir,  Indo che l'om de sil e pe all'ora de la not.

TEST_AUDIO_3:  Event Tramuntena e Sol

TEST_AUDIO_4:  Adesso che arriva, idete la Cristia.



In [ ]:
# N.B. Cell commented out so as not to run it by accident.
# Run this cell to transcribe test set of shortseg dataset.
# transcribe_folder(model=multi_whisp_s_model, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=MULTI_WHISP_S_SHORTSEG)

Found 875 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 875/875 [04:29<00:00,  3.24file/s]


Transcription complete!


In [ ]:
# N.B. Cell commented out so as not to run it by accident.
# Run this cell to transcribe test set of longseg dataset. 
# transcribe_folder(model=multi_whisp_s_model, test_partition_dir=TEST_DIR_LONGSEG, output_dir=MULTI_WHISP_S_LONGSEG)

Found 206 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 206/206 [01:26<00:00,  2.38file/s]


Transcription complete!


In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del multi_whisp_s_model

gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

### 2.4 BuzzASR/Italian

In [ ]:
# Setup/convert BuzzASR
model_dir = setup_asr_model(
    model_name="BuzzASR/italian",
    output_dir=BASE_MODELS_DIR / "buzz_italian_ct2"
)

# Load model
buzz_model = load_asr_model(
    model_path=model_dir,
    device=device,
    compute_type=compute_type
)

Found existing weigths in: /home/ialiani/thesis_Aliani/baseline_models/buzz_italian_ct2


In [ ]:
# Transcription test with single audio files
for i, audio in enumerate(test_audios, 1):
    segments, info = buzz_model.transcribe(audio, beam_size=5, language="it")
    transcription = " ".join(seg.text for seg in segments)
    print(f"TEST_AUDIO_{i}: {transcription}\n")

In [ ]:
# N.B. Cell commented out so as not to run it by accident.
# Run this cell to transcribe test set of shortseg dataset.
# transcribe_folder(model=buzz_model, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=ITA_BUZZ_SHORTSEG)

In [ ]:
# N.B. Cell commented out so as not to run it by accident.
# Run this cell to transcribe test set of longseg dataset.
# transcribe_folder(model=buzz_model, test_partition_dir=TEST_DIR_LONGSEG, output_dir=ITA_BUZZ_LONGSEG)

In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del buzz_model

gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

### 2.5 Litus model

In [ ]:
# Setup/convert litus-small
model_dir = setup_asr_model(
    model_name="litus-ai/whisper-small-ita",
    output_dir=BASE_MODELS_DIR / "litus_small_ita_ct2"
)

# Load model
litus_model = load_asr_model(
    model_path=model_dir,
    device=device,
    compute_type=compute_type
)

In [ ]:
# Transcription test with single audio files
for i, audio in enumerate(test_audios, 1):
    segments, info = litus_model.transcribe(audio, beam_size=5, language="it")
    transcription = " ".join(seg.text for seg in segments)
    print(f"TEST_AUDIO_{i}: {transcription}\n")

In [ ]:
# N.B. Cell commented out so as not to run on accident.
# Run this cell to transcribe test set of shortseg dataset.
# transcribe_folder(model=litus_model, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=ITA_LITUS_SHORTSEG)

In [ ]:
# N.B. Cell commented out so as not to run on accident.
# Run this cell to transcribe test set of longseg dataset.
# transcribe_folder(model=litus_model, test_partition_dir=TEST_DIR_LONGSEG, output_dir=ITA_LITUS_LONGSEG)

In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del litus_model

gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

### 2.6 Sandiago model
Since this is a large model, when working on my laptop I downloaded the weigths from the terminal (same as with BuzzASR). <br>
This is necessary because if I try downloading from the notebook, it gets stuck 1/3 through the download

In [ ]:
# Setup/convert Sandiago21
sandiago_model_dir = setup_asr_model(
    model_name="Sandiago21/whisper-large-v2-italian",
    output_dir=BASE_MODELS_DIR / "sandiago_large_v2_it_ct2"
)

# Load model
sandiago_model = load_asr_model(
    model_path=sandiago_model_dir,
    device=device,
    compute_type=compute_type
)

In [ ]:
# Transcription test with single audio files
for i, audio in enumerate(test_audios, 1):
    segments, info = sandiago_model.transcribe(audio, beam_size=5, language="it")
    transcription = " ".join(seg.text for seg in segments)
    print(f"TEST_AUDIO_{i}: {transcription}\n")

In [ ]:
# N.B. Cell commented out so as not to run on accident.
# Run this cell to transcribe test set of shortseg dataset.
# transcribe_folder(model=sandiago_model, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=ITA_SANDIAGO_SHORTSEG)

In [ ]:
# N.B. Cell commented out so as not to run on accident.
# Run this cell to transcribe test set of longseg dataset.
# transcribe_folder(model=sandiago_model, test_partition_dir=TEST_DIR_LONGSEG, output_dir=ITA_SANDIAGO_LONGSEG)

In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del sandiago_model

gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

## 3. Preparing Data for fine-tuning

1. Transforming dataset into Hugging Face DatasetDict (= building a Hugging Face DatasetDict from pre-processed train, dev, and test folders) using *load_split_to_dict()* which reads paired files into memory and feeds them to Dataset.from_dict(). <br>
- .wav with corresponding .txt are paired by filename 
- wrapped in a DatasetDict
- audio column casted with Audio(sampling_rate=16000) so Hugging Face decodes them on the fly.

N.B.:
>> *.wav files are suppposed to be already 16 kHz, mono, and shorter than 30 s, and transcripts pre-cleaned (lowercased, stripped of punctuation, whitespaces removed) in matching .txt files. **All this was done with pre-processing_pipeline.ipynb notebook**. <br>
>> *Test set is left untouched for zero-shot inference and post-fine-tuning evaluation.

2. Vectorizing data using *prepare_dataset()* (converts that clean raw data into the mathematical matrices and integer tensors required by the neural network architecture). <br>
- Audio: 1D Waveform $\rightarrow$ 2D Log-Mel Spectrogram <br>
Feature_extractor: Whisper’s encoder expects an 80-channel (or 128-channel for v3) log-Mel spectrogram padded or truncated to a fixed $3000$ temporal frames (= 30 seconds). <br>
processor.feature_extractor performs the Short-Time Fourier Transform (STFT), applies the triangular Mel filterbank, takes the log, and pads to 3000 frames. <br>
- Text: String $\rightarrow$ Token IDs <br>
Tokenizer: The neural network’s decoder generates integer indices into Whisper’s vocabulary table (token IDs), not text strings. <br>
Processor.tokenizer maps clean subwords/characters to their corresponding integers. <br>

In [ ]:
# Build DatasetDicts for both longseg and shortseg datasets, containing ONLY train and dev (this step is independent from ASR models)
# Cast audio column so Hugging Face decodes at 16kHz

shortseg_dataset_dict = DatasetDict({
    "train": Dataset.from_dict(load_split_to_dict(SHORTSEG_DATASET / "train")),
    "dev": Dataset.from_dict(load_split_to_dict(SHORTSEG_DATASET / "dev")),
})
shortseg_dataset_dict = shortseg_dataset_dict.cast_column("audio", Audio(sampling_rate=16000))

longseg_dataset_dict = DatasetDict({
    "train": Dataset.from_dict(load_split_to_dict(LONGSEG_DATASET / "train")),
    "dev": Dataset.from_dict(load_split_to_dict(LONGSEG_DATASET / "dev")),
})
longseg_dataset_dict = longseg_dataset_dict.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
# Verification cell that print first two entries in the dataset dictionaries.

example = shortseg_dataset_dict["train"][0]

print("Keys:", example.keys())
print("Sentence:", example["sentence"])
print("Audio metadata:")
print(" - Path:", example["audio"]["path"])
print(" - Sampling rate:", example["audio"]["sampling_rate"])
print(" - Array shape:", example["audio"]["array"].shape)
print(" - Array dtype:", example["audio"]["array"].dtype)
print(" - Duration (seconds):", len(example["audio"]["array"]) / example["audio"]["sampling_rate"])

example = longseg_dataset_dict["train"][0]

print("Keys:", example.keys())
print("Sentence:", example["sentence"])
print("Audio metadata:")
print(" - Path:", example["audio"]["path"])
print(" - Sampling rate:", example["audio"]["sampling_rate"])
print(" - Array shape:", example["audio"]["array"].shape)
print(" - Array dtype:", example["audio"]["array"].dtype)
print(" - Duration (seconds):", len(example["audio"]["array"]) / example["audio"]["sampling_rate"])

In [ ]:
# Verification cell n.2
# Count actual .wav files physically present on disk
disk_counts = {split: len(list((LONGSEG_DATASET / split).glob("*.wav"))) for split in ["train", "dev"]}

# Count samples registered in the DatasetDict
dict_counts = {split: len(longseg_dataset_dict[split]) for split in ["train", "dev"]}

print("=== PARTITION VERIFICATION ===")
for split in ["train", "dev"]:
    on_disk = disk_counts[split]
    in_dict = dict_counts[split]
    status = "OK" if on_disk == in_dict else "MISMATCH"
    print(f"[{split.upper()}] Disk: {on_disk} | DatasetDict: {in_dict} -> {status}")

# Check for any missing companion .txt files or empty transcripts on disk
missing_txt = []
empty_txt = []

for split in ["train", "dev"]:
    for wav_path in (LONGSEG_DATASET / split).glob("*.wav"):
        txt_path = wav_path.with_suffix(".txt")
        if not txt_path.exists():
            missing_txt.append(str(wav_path))
        else:
            with open(txt_path, "r", encoding="utf-8") as f:
                if not f.read().strip():
                    empty_txt.append(str(txt_path))

print(f"\nMissing .txt files: {len(missing_txt)}")
print(f"Empty transcripts:  {len(empty_txt)}")

if not missing_txt and not empty_txt and all(disk_counts[s] == dict_counts[s] for s in ["train", "dev"]):
    print("\nAll audio files on disk are 1:1 matched, transcribed, and loaded into raw_dataset.")

### 3.1 Configuring processors

Feature Extractor (Mel Bins):
* openai/whisper-large-v3 and BuzzASR use 128 mel frequency bins.
* Whisper Small, Whisper Medium, Litus-ai, and Sandiago21 use 80 mel frequency bins.

N.B. To ensure that model configs match their processors exactly, each model is initialized cleanly from their hub repositories.

In [ ]:
processor_whisp_large = WhisperProcessor.from_pretrained(
    "openai/whisper-large-v3",
    language="italian",
    task="transcribe",
    cache_dir=USER_CACHE
)

In [7]:
processor_whisp_medium = WhisperProcessor.from_pretrained(
    "openai/whisper-medium",
    language="italian",
    task="transcribe",
    cache_dir=USER_CACHE
)

In [5]:
processor_whisp_small = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="italian",
    task="transcribe",
    cache_dir=USER_CACHE
)

In [ ]:
processor_buzz = WhisperProcessor.from_pretrained(
    "BuzzASR/italian",
    language="italian",
    task="transcribe",
    use_fast=False,
    cache_dir=USER_CACHE
)

In [6]:
processor_litus = WhisperProcessor.from_pretrained(
    "litus-ai/whisper-small-ita",
    language="italian",
    task="transcribe",
    cache_dir=USER_CACHE
)

In [5]:
processor_sandiago = WhisperProcessor.from_pretrained(
    "Sandiago21/whisper-large-v2-italian",
    language="italian",
    task="transcribe",
    cache_dir=USER_CACHE
)
# Register all Whisper control tags as special tokens so skip_special_tokens=True strips them
processor_sandiago.tokenizer.add_special_tokens({
    "additional_special_tokens": [t for t in processor_sandiago.tokenizer.get_vocab() if t.startswith("<|")]
})

106

### 3.2 Vectorized datasets 
Vectorization of *shortseg* and *longseg* training and development partitions is carried out for each model separately, to avoid incompatibilities. 

In [ ]:
# Whisper Large
vectorized_shortseg_dataset_whisp_large = shortseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_whisp_large},
    remove_columns=shortseg_dataset_dict.column_names["train"],
    num_proc=1
)

vectorized_longseg_dataset_whisp_large = longseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_whisp_large},
    remove_columns=longseg_dataset_dict.column_names["train"],
    num_proc=1
)

In [ ]:
# Whisper Medium
vectorized_shortseg_dataset_whisp_medium = shortseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_whisp_medium},
    remove_columns=shortseg_dataset_dict.column_names["train"],
    num_proc=1
)

vectorized_longseg_dataset_whisp_medium = longseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_whisp_medium},
    remove_columns=longseg_dataset_dict.column_names["train"],
    num_proc=1
)

Map:   0%|          | 0/3180 [00:00<?, ? examples/s]

Map:   0%|          | 0/569 [00:00<?, ? examples/s]

Map:   0%|          | 0/708 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [ ]:
# Whisper Small
vectorized_shortseg_dataset_whisp_small = shortseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_whisp_small},
    remove_columns=shortseg_dataset_dict.column_names["train"],
    num_proc=1
)

vectorized_longseg_dataset_whisp_small = longseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_whisp_small},
    remove_columns=longseg_dataset_dict.column_names["train"],
    num_proc=1
)

Map:   0%|          | 0/3180 [00:00<?, ? examples/s]

Map:   0%|          | 0/569 [00:00<?, ? examples/s]

Map:   0%|          | 0/708 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [ ]:
# BuzzASR
vectorized_shortseg_dataset_buzz = shortseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_buzz},
    remove_columns=shortseg_dataset_dict.column_names["train"],
    num_proc=1
)

vectorized_longseg_dataset_buzz = longseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_buzz},
    remove_columns=longseg_dataset_dict.column_names["train"],
    num_proc=1
)

Map:   0%|          | 0/3180 [00:00<?, ? examples/s]

Map:   0%|          | 0/569 [00:00<?, ? examples/s]

Map:   0%|          | 0/708 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [ ]:
# Litus-ai
vectorized_shortseg_dataset_litus = shortseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_litus},
    remove_columns=shortseg_dataset_dict.column_names["train"],
    num_proc=1
)

vectorized_longseg_dataset_litus = longseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_litus},
    remove_columns=longseg_dataset_dict.column_names["train"],
    num_proc=1
)

Map:   0%|          | 0/3180 [00:00<?, ? examples/s]

Map:   0%|          | 0/569 [00:00<?, ? examples/s]

Map:   0%|          | 0/708 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [ ]:
# Sandiago21
vectorized_shortseg_dataset_sandiago = shortseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_sandiago},
    remove_columns=shortseg_dataset_dict.column_names["train"],
    num_proc=1
)

vectorized_longseg_dataset_sandiago = longseg_dataset_dict.map(
    prepare_dataset,
    fn_kwargs={"processor": processor_sandiago},
    remove_columns=longseg_dataset_dict.column_names["train"],
    num_proc=1
)

Map:   0%|          | 0/3180 [00:00<?, ? examples/s]

Map:   0%|          | 0/569 [00:00<?, ? examples/s]

Map:   0%|          | 0/708 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

### 3.5 Collators
In this section, for each model, the collator is instatiated with the processor for that model.

In [ ]:
data_collator_whisp_large = DataCollatorSpeechSeq2SeqWithPadding(processor=processor_whisp_large)

In [9]:
data_collator_whisp_medium = DataCollatorSpeechSeq2SeqWithPadding(processor=processor_whisp_medium)

In [7]:
data_collator_whisp_small = DataCollatorSpeechSeq2SeqWithPadding(processor=processor_whisp_small)

In [10]:
data_collator_buzz = DataCollatorSpeechSeq2SeqWithPadding(processor=processor_buzz)

In [8]:
data_collator_litus = DataCollatorSpeechSeq2SeqWithPadding(processor=processor_litus)

In [7]:
data_collator_sandiago = DataCollatorSpeechSeq2SeqWithPadding(processor=processor_sandiago)

In [ ]:
# Verification cell
# Examples using longseg vectorized dataset for multilingual Whisper and BuzzASR model.
# Grab 2 samples from train to simulate a small batch
sample_batch = [vectorized_longseg_dataset_multi_buzz["train"][0], vectorized_longseg_dataset_multi_buzz["train"][1]]
collated = data_collator_multi_buzz(sample_batch)

print("Batch input_features shape:", collated["input_features"].shape)
print("Batch labels shape:", collated["labels"].shape)
print("Masked padding tokens (-100) in labels:", (collated["labels"] == -100).sum().item())

# Example using shortseg vectorized dataset for litus model
sample_batch = [vectorized_shortseg_dataset_litus["train"][0], vectorized_shortseg_dataset_litus["train"][1]]
collated = data_collator_litus(sample_batch)

print("\nBatch input_features shape:", collated["input_features"].shape)
print("Batch labels shape:", collated["labels"].shape)
print("Masked padding tokens (-100) in labels:", (collated["labels"] == -100).sum().item())

# Example using longseg vectorized dataset for sandiago model
sample_batch = [vectorized_shortseg_dataset_sandiago["train"][0], vectorized_shortseg_dataset_sandiago["train"][1]]
collated = data_collator_sandiago(sample_batch)

print("\nBatch input_features shape:", collated["input_features"].shape)
print("Batch labels shape:", collated["labels"].shape)
print("Masked padding tokens (-100) in labels:", (collated["labels"] == -100).sum().item())

### 3.6 Compute metrics
In this section: Computing metrics are instatiated for each model.

Following the Hugging Face chapter on fine-tuning (https://huggingface.co/learn/audio-course/chapter5/fine-tuning) defining the evaluation metric function (compute_metrics with Word Error Rate) so the trainer can track validation performance on dev partition. <br>
common critique in the literature is actually that WER is too high and disproportionately punishing, which obscures what the model is learning. <br>

Standard practice is to compute and report both WER and CER side-by-side: <br>   
* Character Error Rate (CER) serves as primary diagnostic for acoustic fidelity and phonetic transcription quality.
* Word Error Rate (WER) serves as the standard benchmark for downstream usability and comparison with external literature.   

In [9]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

In [ ]:
compute_metrics_whisp_large = build_compute_metrics(processor_whisp_large.tokenizer)

In [11]:
compute_metrics_whisp_medium = build_compute_metrics(processor_whisp_medium.tokenizer)

In [9]:
compute_metrics_whisp_small = build_compute_metrics(processor_whisp_small.tokenizer)

In [12]:
compute_metrics_buzz = build_compute_metrics(processor_buzz.tokenizer)

In [10]:
compute_metrics_litus = build_compute_metrics(processor_litus.tokenizer)

In [9]:
compute_metrics_sandiago = build_compute_metrics(processor_sandiago.tokenizer)

## 4. Model Fine-tuning <br>

Fine-tuning is carried out on the 6 models using the vectorized training and development partitions to adapt the models specifically for Romagnol speech. This configuration aims to enhance the models’ performance by making them more sensitive to Romagnol-specific phonetic and linguistic patterns. <br>

N.B. Dual-experimental setup to investigate impact of longer vs shorter average audio-clip duration > each model is trained and evaluated on both the *shortseg* and *longseg* dataset. <br>

Fine-Tuning Hyperparameter Configurations and Optimization Arguments

| Hyperparameter | Hugging Face Argument | Small Models | Large Models |
| :--- | :--- | :---: | :---: |
| **Architecture-Specific Configuration** | | | |
| Per-device train batch size | `per_device_train_batch_size` | 8 | 2 |
| Gradient accumulation steps | `gradient_accumulation_steps` | 4 | 16 |
| Effective batch size | — | 32 | 32 |
| Per-device eval batch size | `per_device_eval_batch_size` | 8 | 2 |
| Learning rate ($\eta$) | `learning_rate` | $5.0 \times 10^{-5}$ | $2.5 \times 10^{-6}$ |
| Warmup steps | `warmup_steps` | 300 | 150 |
| Maximum training steps | `max_steps` | 2000 | 1000 |
| **Shared Optimization, Regularization & Evaluation Setup** | | | |
| Optimizer | `optim` | `adamw_bnb_8bit` | `adamw_bnb_8bit` |
| Precision | `fp16` | True (Mixed Precision) | True (Mixed Precision) |
| Gradient checkpointing | `gradient_checkpointing` | Enabled | Enabled |
| Weight decay | `weight_decay` | 0.01 | 0.01 |
| Learning rate schedule | `lr_scheduler_type` | Linear decay | Linear decay |
| Random seed / Data seed | `seed` / `data_seed` | 42 | 42 |
| Evaluation strategy | `eval_strategy` | Steps (every 100 steps) | Steps (every 100 steps) |
| Logging interval | `logging_steps` | Every 25 steps | Every 25 steps |
| Evaluation / Save frequency | `eval_steps` / `save_steps` | Every 100 steps | Every 100 steps |
| Checkpoint retention limit | `save_total_limit` | 2 | 2 |
| Model selection metric | `metric_for_best_model` | Character Error Rate (CER) | Character Error Rate (CER) |
| Best model criterion | `greater_is_better` | False (Minimization) | False (Minimization) |
| Autoregressive generation | `predict_with_generate` | Enabled (`max_length=225`) | Enabled (`max_length=225`) |
| Experiment logging backend | `report_to` | `tensorboard` | `tensorboard` |

### 4.1 Multilingual Whisper Large

In [ ]:
# Load model
multi_whisp_l_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-large-v3", 
    cache_dir=USER_CACHE
).float()

multi_whisp_l_model.enable_input_require_grads() # Hook for gradient checkpointing (not needed because this pipeline does full fine-tuning) (not needed because this pipeline does full fine-tuning)
multi_whisp_l_model.generation_config.language = "italian"
multi_whisp_l_model.generation_config.task = "transcribe"
multi_whisp_l_model.generation_config.forced_decoder_ids = None
multi_whisp_l_model.generation_config.suppress_tokens = None
multi_whisp_l_model.generation_config.begin_suppress_tokens = None

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [25]:
multi_whisp_l_args = get_training_args("multi_whisp_l", is_small_model=False)

In [26]:
multi_whisp_l_shortseg_trainer = Seq2SeqTrainer(
    args=multi_whisp_l_args["shortseg"],
    model=multi_whisp_l_model,
    train_dataset=vectorized_shortseg_dataset_whisp_large["train"],
    eval_dataset=vectorized_shortseg_dataset_whisp_large["dev"],
    data_collator=data_collator_whisp_large,
    compute_metrics=compute_metrics_whisp_large,
    processing_class=processor_whisp_large.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [27]:
# Run this cell to test the baseline, pre-trained Whisper Large-v3 weights directly out of the box on the dev partition.
baseline_metrics = multi_whisp_l_shortseg_trainer.evaluate()

Training Loss,Validation Loss,Step,Wer,Cer
No log,4.016860,0,90.014144,47.071678


In [28]:
train_result_multi_whisp_l_shortseg = multi_whisp_l_shortseg_trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
100,28.513411,1.667546,73.861386,26.985514
200,13.143281,0.900577,63.309760,22.783467
300,10.138612,0.820579,69.448373,26.080170
400,7.195764,0.818395,62.715700,22.265235
500,5.435470,0.841903,63.338048,21.859391
600,4.330194,0.889336,63.168317,21.909341
700,3.062153,0.958346,77.991513,33.784965


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


In [ ]:
final_model_multi_whisp_l_shortseg = FINAL_MODELS_DIR / "multi_whisp_l_shortseg"
final_model_multi_whisp_l_shortseg.mkdir(parents=True, exist_ok=True)

# Save best multi_whisp weights and processor
multi_whisp_l_shortseg_trainer.save_model(str(final_model_multi_whisp_l_shortseg))
processor_whisp_large.save_pretrained(str(final_model_multi_whisp_l_shortseg))
print(f"multi_whisp best model successfully exported to: {final_model_multi_whisp_l_shortseg}")

# Clean up bulky intermediate checkpoints from multi_whisp/shortseg to free disk
ckpt_base = Path.home() / "thesis_Aliani" / "checkpoints" / "multi_whisp_l" / "shortseg"
for item in ckpt_base.glob("checkpoint-*"):
    if item.is_dir():
        shutil.rmtree(item)
        print(f"Deleted checkpoint to save disk space: {item.name}")

# Completely flush VRAM so GPU 0 is fresh for the next run
del multi_whisp_l_shortseg_trainer
del multi_whisp_l_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

multi_whisp best model successfully exported to: /home/ialiani/thesis_Aliani/final_models/multi_whisp_l_shortseg
Deleted checkpoint to save disk space: checkpoint-700
Deleted checkpoint to save disk space: checkpoint-500
GPU memory cleared.


### 4.2 Multilingual Whisper medium

In [ ]:
# Load model
multi_whisp_m_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-medium", 
    cache_dir=USER_CACHE
).float()

# Hook for gradient checkpointing (not needed because this pipeline does full fine-tuning)
multi_whisp_m_model.enable_input_require_grads()
multi_whisp_m_model.generation_config.language = "italian"
multi_whisp_m_model.generation_config.task = "transcribe"
multi_whisp_m_model.generation_config.forced_decoder_ids = None
multi_whisp_m_model.generation_config.suppress_tokens = None
multi_whisp_m_model.generation_config.begin_suppress_tokens = None

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

In [33]:
multi_whisp_m_args = get_training_args("multi_whisp_m", is_small_model=False)

In [34]:
multi_whisp_m_longseg_trainer = Seq2SeqTrainer(
    args=multi_whisp_m_args["longseg"],
    model=multi_whisp_m_model,
    train_dataset=vectorized_longseg_dataset_whisp_medium["train"],
    eval_dataset=vectorized_longseg_dataset_whisp_medium["dev"],
    data_collator=data_collator_whisp_medium,
    compute_metrics=compute_metrics_whisp_medium,
    processing_class=processor_whisp_medium.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
# Run this cell to test the baseline, pre-trained Whisper Large-v3 weights directly out of the box on the dev partition.
baseline_metrics = multi_whisp_m_longseg_trainer.evaluate()

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Training Loss,Validation Loss,Step,Wer,Cer
No log,6.586172,0,100.028289,59.153347


In [35]:
train_result_multi_whisp_m_longseg = multi_whisp_m_longseg_trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
100,28.502400,1.758642,70.851004,26.582202
200,11.064078,1.160860,61.549336,20.289767
300,3.577592,1.282203,58.778626,19.047042
400,1.007623,1.507573,59.344077,19.283463
500,0.248367,1.652750,60.418434,20.356450


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


In [ ]:
final_model_multi_whisp_m_longseg = FINAL_MODELS_DIR / "multi_whisp_m_longseg"
final_model_multi_whisp_m_longseg.mkdir(parents=True, exist_ok=True)

# Save best multi_whisp weights and processor
multi_whisp_m_longseg_trainer.save_model(str(final_model_multi_whisp_m_longseg))
processor_whisp_medium.save_pretrained(str(final_model_multi_whisp_m_longseg))
print(f"multi_whisp best model successfully exported to: {final_model_multi_whisp_m_longseg}")

# Clean up bulky intermediate checkpoints from multi_whisp/shortseg to free disk
ckpt_base = Path.home() / "thesis_Aliani" / "checkpoints" / "multi_whisp_m" / "longseg"
for item in ckpt_base.glob("checkpoint-*"):
    if item.is_dir():
        shutil.rmtree(item)
        print(f"Deleted checkpoint to save disk space: {item.name}")

# Completely flush VRAM so GPU 0 is fresh for the next run
del multi_whisp_m_longseg_trainer
del multi_whisp_m_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

multi_whisp best model successfully exported to: /home/ialiani/thesis_Aliani/final_models/multi_whisp_m_longseg
Deleted checkpoint to save disk space: checkpoint-500
Deleted checkpoint to save disk space: checkpoint-300
GPU memory cleared.


### 4.3 Multilingual Whisper small

In [ ]:
# Load model
multi_whisp_s_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small", 
    cache_dir=USER_CACHE
).float()

# Hook for gradient checkpointing (not needed because this pipeline does full fine-tuning)
multi_whisp_s_model.enable_input_require_grads()
multi_whisp_s_model.generation_config.language = "italian"
multi_whisp_s_model.generation_config.task = "transcribe"
multi_whisp_s_model.generation_config.forced_decoder_ids = None
multi_whisp_s_model.generation_config.suppress_tokens = None
multi_whisp_s_model.generation_config.begin_suppress_tokens = None

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [32]:
multi_whisp_s_args = get_training_args("multi_whisp_s", is_small_model=True)

In [ ]:
multi_whisp_s_shortseg_trainer = Seq2SeqTrainer(
    args=multi_whisp_s_args["shortseg"],
    model=multi_whisp_s_model,
    train_dataset=vectorized_shortseg_dataset_whisp_small["train"],
    eval_dataset=vectorized_shortseg_dataset_whisp_small["dev"],
    data_collator=data_collator_whisp_small,
    compute_metrics=compute_metrics_whisp_small,
    processing_class=processor_whisp_small.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [23]:
# Run this cell to test the baseline, pre-trained Whisper Large-v3 weights directly out of the box on the dev partition.
baseline_metrics = multi_whisp_s_shortseg_trainer.evaluate()

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Training Loss,Validation Loss,Step,Wer,Cer
No log,6.954328,0,108.373409,66.289960


In [ ]:
train_result_multi_whisp_s_shortseg = multi_whisp_s_shortseg_trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
100,3.526076,1.268468,59.117897,19.629001
200,0.331069,1.327824,59.966073,20.871726
300,0.145566,1.456765,56.771275,18.634821
400,0.117267,1.513301,56.205824,18.234724
500,0.080342,1.578087,54.650834,17.640640
600,0.051691,1.638299,53.661295,17.379971
700,0.034366,1.692077,55.018377,18.295344
800,0.026302,1.744630,54.085383,17.343598
900,0.017898,1.740399,55.188012,17.846751
1000,0.012709,1.781408,55.244558,18.374151


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


In [ ]:
final_model_multi_whisp_s_shortseg = FINAL_MODELS_DIR / "multi_whisp_s_shortseg"
final_model_multi_whisp_s_shortseg.mkdir(parents=True, exist_ok=True)

# Save best multi_whisp weights and processor
multi_whisp_s_shortseg_trainer.save_model(str(final_model_multi_whisp_s_shortseg))
processor_whisp_small.save_pretrained(str(final_model_multi_whisp_s_shortseg))
print(f"multi_whisp best model successfully exported to: {final_model_multi_whisp_s_shortseg}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

multi_whisp best model successfully exported to: /home/ialiani/thesis_Aliani/final_models/multi_whisp_s_longseg


In [ ]:
# Clean up bulky intermediate checkpoints from multi_whisp/shortseg to free disk
ckpt_base = Path.home() / "thesis_Aliani" / "checkpoints" / "multi_whisp_s" / "longseg"
for item in ckpt_base.glob("checkpoint-*"):
    if item.is_dir():
        shutil.rmtree(item)
        print(f"Deleted checkpoint to save disk space: {item.name}")

# Completely flush VRAM so GPU 0 is fresh for the next run
del multi_whisp_s_longseg_trainer
del multi_whisp_s_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

Deleted checkpoint to save disk space: checkpoint-800
Deleted checkpoint to save disk space: checkpoint-1000
GPU memory cleared.


### 4.4 BuzzASR/Italian

In [ ]:
buzz_model = WhisperForConditionalGeneration.from_pretrained(
    "BuzzASR/italian", 
    cache_dir=USER_CACHE
).float()

buzz_model.enable_input_require_grads()

# Restore default suppressions from openai/whisper-large-v3
base_gen_config = GenerationConfig.from_pretrained(
    "openai/whisper-large-v3", 
    cache_dir=USER_CACHE
)
buzz_model.generation_config.suppress_tokens = base_gen_config.suppress_tokens
buzz_model.generation_config.begin_suppress_tokens = base_gen_config.begin_suppress_tokens

# Model-level generation cap & anti-loop settings (no changes to get_training_args needed!)
buzz_model.generation_config.max_length = 128
buzz_model.generation_config.repetition_penalty = 1.2
buzz_model.generation_config.no_repeat_ngram_size = 3
buzz_model.generation_config.return_timestamps = False
buzz_model.generation_config.language = "italian"
buzz_model.generation_config.task = "transcribe"
buzz_model.generation_config.forced_decoder_ids = None

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [21]:
buzz_args = get_training_args("buzz", is_small_model=False)

In [22]:
def _safe_batch_decode(sequences, *args, **kwargs):
    # Handle NumPy ndarrays (passed directly from Trainer evaluation_loop)
    if isinstance(sequences, np.ndarray):
        sequences = np.where(sequences == -100, processor_buzz.tokenizer.pad_token_id, sequences)
    # Handle lists of arrays or nested lists
    elif isinstance(sequences, (list, tuple)) and len(sequences) > 0:
        if isinstance(sequences[0], np.ndarray):
            sequences = [np.where(seq == -100, processor_buzz.tokenizer.pad_token_id, seq) for seq in sequences]
        elif isinstance(sequences[0], (list, tuple)):
            pad_id = processor_buzz.tokenizer.pad_token_id
            sequences = [[pad_id if tok == -100 else tok for tok in seq] for seq in sequences]
            
    return _orig_batch_decode(sequences, *args, **kwargs)

In [ ]:
processor_buzz.tokenizer.clean_up_tokenization_spaces = False
# If the model has an EOS token, you can often use it as PAD/UNK fallback
if processor_buzz.tokenizer.pad_token is None:
    processor_buzz.tokenizer.pad_token = processor_buzz.tokenizer.eos_token
    processor_buzz.tokenizer.pad_token_id = processor_buzz.tokenizer.eos_token_id

if processor_buzz.tokenizer.unk_token is None:
    processor_buzz.tokenizer.unk_token = processor_buzz.tokenizer.eos_token
    processor_buzz.tokenizer.unk_token_id = processor_buzz.tokenizer.eos_token_id

# Instantiate the trainer
buzz_longseg_trainer = Seq2SeqTrainer(
    args=buzz_args["longseg"],
    model=buzz_model,
    train_dataset=vectorized_longseg_dataset_buzz["train"],
    eval_dataset=vectorized_longseg_dataset_buzz["dev"],
    data_collator=data_collator_buzz,
    compute_metrics=compute_metrics_buzz,
    tokenizer=processor_buzz.tokenizer,
    processing_class=processor_buzz.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
# Store original method
_orig_batch_decode = processor_buzz.tokenizer.batch_decode

# Apply wrapper directly to the tokenizer instance used by compute_metrics_buzz
processor_buzz.tokenizer.batch_decode = _safe_batch_decode

TypeError: Seq2SeqTrainer.__init__() got an unexpected keyword argument 'tokenizer'

In [17]:
# Run this cell to test the baseline, pre-trained Whisper Large-v3 weights directly out of the box on the dev partition.
baseline_metrics = buzz_shortseg_trainer.evaluate()

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Training Loss,Validation Loss,Step,Wer,Cer
No log,2.509815,0,89.420085,40.578172


In [26]:
train_result_buzz_longseg = buzz_longseg_trainer.train()

Step,Training Loss,Validation Loss


RecursionError: maximum recursion depth exceeded while calling a Python object

In [ ]:
final_model_buzz_longseg = FINAL_MODELS_DIR / "buzz_longseg"
final_model_buzz_longseg.mkdir(parents=True, exist_ok=True)

# Save best multi_whisp weights and processor
buzz_longseg_trainer.save_model(str(final_model_buzz_longseg))
processor_buzz.save_pretrained(str(final_model_buzz_longseg))
print(f"buzz best model successfully exported to: {final_model_buzz_longseg}")

# Clean up bulky intermediate checkpoints from multi_whisp/shortseg to free disk
ckpt_base = Path.home() / "thesis_Aliani" / "checkpoints" / "buzz" / "longseg"
for item in ckpt_base.glob("checkpoint-*"):
    if item.is_dir():
        shutil.rmtree(item)
        print(f"Deleted checkpoint to save disk space: {item.name}")

# Completely flush VRAM so GPU 0 is fresh for the next run
del buzz_longseg_trainer
del buzz_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

buzz best model successfully exported to: /home/ialiani/thesis_Aliani/final_models/buzz_shortseg
Deleted checkpoint to save disk space: checkpoint-200
Deleted checkpoint to save disk space: checkpoint-400
GPU memory cleared.


### 4.5 Litus model

In [ ]:
# Load model
litus_model = WhisperForConditionalGeneration.from_pretrained(
    "litus-ai/whisper-small-ita", 
    cache_dir=USER_CACHE
).float()

# Hook for gradient checkpointing (not needed because this pipeline does full fine-tuning)
litus_model.enable_input_require_grads()

litus_model.generation_config.language = "italian"
litus_model.generation_config.task = "transcribe"
litus_model.generation_config.forced_decoder_ids = None
litus_model.generation_config.suppress_tokens = None
litus_model.generation_config.begin_suppress_tokens = None
litus_model.generation_config.repetition_penalty = 1.2
litus_model.generation_config.no_repeat_ngram_size = 3

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [12]:
litus_args = get_training_args("litus", is_small_model=True)

In [13]:
litus_shortseg_trainer = Seq2SeqTrainer(
    args=litus_args["shortseg"],
    model=litus_model,
    train_dataset=vectorized_shortseg_dataset_litus["train"],
    eval_dataset=vectorized_shortseg_dataset_litus["dev"],
    data_collator=data_collator_litus,
    compute_metrics=compute_metrics_litus,
    processing_class=processor_litus.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
# Run this cell to test the baseline, pre-trained Whisper Large-v3 weights directly out of the box on the dev partition.
baseline_metrics = litus_longseg_trainer.evaluate()

Training Loss,Validation Loss,Step,Wer,Cer
No log,3.666894,0,94.766620,57.011738


In [14]:
train_result_litus_shortseg = litus_shortseg_trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
100,4.787492,1.143827,70.806223,25.443307
200,3.086924,0.977124,63.224894,22.539960
300,1.926184,0.990287,60.480905,20.960290
400,1.100006,1.066402,60.056577,20.985265
500,0.535523,1.089127,58.274399,20.504496
600,0.350567,1.158387,57.595474,20.510739
700,0.204887,1.154935,56.859972,20.223526
800,0.181681,1.181817,57.538897,20.198551
900,0.135281,1.209809,55.162659,19.493007
1000,0.090648,1.233589,54.568600,19.180819


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (13 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


In [15]:
final_model_litus_shortseg = FINAL_MODELS_DIR / "litus_shortseg"
final_model_litus_shortseg.mkdir(parents=True, exist_ok=True)

# Save best multi_whisp weights and processor
litus_shortseg_trainer.save_model(str(final_model_litus_shortseg))
processor_litus.save_pretrained(str(final_model_litus_shortseg))
print(f"litus best model successfully exported to: {final_model_litus_shortseg}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

litus best model successfully exported to: /home/ialiani/thesis_Aliani/final_models/litus_shortseg


In [ ]:
# Run this cell if during training evaluation steps results were not printed.
log_history = None

# First choice: read directly from the in-memory trainer instance
if "litus_shortseg_trainer" in globals() and hasattr(
    litus_shortseg_trainer, "state"
):
    log_history = litus_shortseg_trainer.state.log_history

# Second choice: look in the trainer's output_dir and checkpoint folders
if not log_history:
    search_dirs = []
    if "litus_shortseg_trainer" in globals() and hasattr(
        litus_shortseg_trainer, "args"
    ):
        search_dirs.append(Path(litus_shortseg_trainer.args.output_dir))

    # Add default intermediate training output directories if standard
    search_dirs.extend(
        [
            Path("./output"),
            Path("./results"),
            Path.home() / "thesis_Aliani" / "output",
            Path.home() / "thesis_Aliani" / "results",
        ]
    )

    for out_dir in search_dirs:
        if not out_dir.exists():
            continue
        # Direct check in output dir
        state_file = out_dir / "trainer_state.json"
        if state_file.exists():
            with open(state_file, "r", encoding="utf-8") as f:
                log_history = json.load(f).get("log_history", [])
            break

        # Check inside checkpoints (checkpoint-*)
        checkpoints = sorted(
            out_dir.glob("checkpoint-*"),
            key=lambda p: int(p.name.split("-")[-1])
            if p.name.split("-")[-1].isdigit()
            else 0,
        )
        if checkpoints:
            latest_ckpt_state = checkpoints[-1] / "trainer_state.json"
            if latest_ckpt_state.exists():
                with open(latest_ckpt_state, "r", encoding="utf-8") as f:
                    log_history = json.load(f).get("log_history", [])
                break

if not log_history:
    raise FileNotFoundError(
        "Could not locate trainer_state.json. If the kernel was restarted, "
        "set `output_dir = Path('/path/to/training_output_dir')` where checkpoints were saved."
    )

# Build metric summary table
train_losses = {
    x["step"]: x["loss"] for x in log_history if "loss" in x and "step" in x
}

rows = []
for entry in log_history:
    if "eval_loss" in entry:
        step = entry.get("step")

        train_loss = train_losses.get(step)
        if train_loss is None and train_losses:
            valid_steps = [s for s in train_losses.keys() if s <= step]
            if valid_steps:
                train_loss = train_losses[max(valid_steps)]

        rows.append(
            {
                "Step": step,
                "Training Loss": (
                    round(train_loss, 4) if train_loss is not None else "-"
                ),
                "Validation Loss": round(entry.get("eval_loss", 0.0), 4),
                "WER": round(
                    entry.get("eval_wer", entry.get("eval_wer_score", 0.0)), 4
                ),
                "CER": round(
                    entry.get("eval_cer", entry.get("eval_cer_score", 0.0)), 4
                ),
            }
        )

df_metrics = pd.DataFrame(rows).sort_values("Step").reset_index(drop=True)
display(df_metrics)

,Step,Training Loss,Validation Loss,WER,CER
0,100,4.7868,1.1437,70.6648,25.4433
1,200,3.0924,0.9786,63.6492,22.7522
2,300,1.9158,0.9777,60.0283,20.8042
3,400,1.0649,1.0554,60.0283,20.8729
4,500,0.5595,1.0975,57.0014,19.9301
5,600,0.3540,1.1297,58.4441,20.3921
6,700,0.2296,1.1844,57.9632,21.0852


In [ ]:
# Clean up bulky intermediate checkpoints from multi_whisp/shortseg to free disk
ckpt_base = Path.home() / "thesis_Aliani" / "checkpoints" / "litus" / "shortseg"
for item in ckpt_base.glob("checkpoint-*"):
    if item.is_dir():
        shutil.rmtree(item)
        print(f"Deleted checkpoint to save disk space: {item.name}")

# Completely flush VRAM so GPU 0 is fresh for the next run
del litus_shortseg_trainer
del litus_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

Deleted checkpoint to save disk space: checkpoint-1500
Deleted checkpoint to save disk space: checkpoint-1700
GPU memory cleared.


### 4.6 Sandiago model

In [ ]:
# Load model in float32
sandiago_model = WhisperForConditionalGeneration.from_pretrained(
    "Sandiago21/whisper-large-v2-italian", 
    cache_dir=USER_CACHE
).float()

# Hook for gradient checkpointing (not needed because this pipeline does full fine-tuning)
sandiago_model.enable_input_require_grads()
sandiago_model.generation_config.language = "italian"
sandiago_model.generation_config.task = "transcribe"
sandiago_model.generation_config.forced_decoder_ids = (
    processor_sandiago.get_decoder_prompt_ids(language="it", task="transcribe")
)
sandiago_model.generation_config.suppress_tokens = None
sandiago_model.generation_config.begin_suppress_tokens = None
sandiago_model.generation_config.repetition_penalty = 1.2
sandiago_model.generation_config.no_repeat_ngram_size = 3

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [31]:
sandiago_args = get_training_args("sandiago", is_small_model=False)

In [32]:
sandiago_shortseg_trainer = Seq2SeqTrainer(
    args=sandiago_args["shortseg"],
    model=sandiago_model,
    train_dataset=vectorized_shortseg_dataset_sandiago["train"],
    eval_dataset=vectorized_shortseg_dataset_sandiago["dev"],
    data_collator=data_collator_sandiago,
    compute_metrics=compute_metrics_sandiago,
    processing_class=processor_sandiago.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [13]:
# Run this cell to test the baseline, pre-trained Whisper Large-v3 weights directly out of the box on the dev partition.
baseline_metrics = sandiago_longseg_trainer.evaluate()

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Training Loss,Validation Loss,Step,Wer,Cer
No log,3.391334,0,90.302516,50.333414


In [33]:
train_result_sandiago_shortseg = sandiago_shortseg_trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
100,26.424229,1.482507,83.224894,32.729770
200,17.427119,1.084133,70.495050,25.462038
300,13.383531,0.978296,66.987270,23.364136
400,10.487106,0.943975,63.564356,22.109141
500,8.810240,0.942062,61.867044,21.353646
600,6.960261,0.971942,61.471004,21.266234
700,5.508387,0.996966,62.093352,21.372378
800,4.481507,1.043928,61.895332,21.509740


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


In [ ]:
final_model_sandiago_shortseg = FINAL_MODELS_DIR / "sandiago_shortseg"
final_model_sandiago_shortseg.mkdir(parents=True, exist_ok=True)

# Save best multi_whisp weights and processor
sandiago_shortseg_trainer.save_model(str(final_model_sandiago_shortseg))
processor_sandiago.save_pretrained(str(final_model_sandiago_shortseg))
print(f"sandiago best model successfully exported to: {final_model_sandiago_shortseg}")

# Clean up bulky intermediate checkpoints from multi_whisp/shortseg to free disk
ckpt_base = Path.home() / "thesis_Aliani" / "checkpoints" / "sandiago" / "shortseg"
for item in ckpt_base.glob("checkpoint-*"):
    if item.is_dir():
        shutil.rmtree(item)
        print(f"Deleted checkpoint to save disk space: {item.name}")

# Completely flush VRAM so GPU 0 is fresh for the next run
del sandiago_shortseg_trainer
del sandiago_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

sandiago best model successfully exported to: /home/ialiani/thesis_Aliani/final_models/sandiago_shortseg
Deleted checkpoint to save disk space: checkpoint-800
Deleted checkpoint to save disk space: checkpoint-600
GPU memory cleared.


## 5. Predictions on test set after fine-tuning to Romagnol

**Structure of the folder containing the final_models weights and configuration**

N.B. Fine-tuning was carried out three times for each models and saved checkpoints were three in total, which was used to transcribe the test set again, then predictions were evaluated and mean ± standard deviation were calculated.
In the final_models folder only CT2 weights were kept, together with first seed models for best-performing model.

```text
thesis_Aliani/
|
└── final_models/
    ├── buzz_longseg_ct2_1/     # Converted CTranslate2 model (model.bin, vocabulary.json, etc.)    
    ├── buzz_shortseg_ct2_1/         
    ├── buzz_shortseg_ct2_2/
    ├── buzz_shortseg_ct2_3/              
    ├── litus_longseg_ct2_1/     
    ├── litus_longseg_ct2_2/
    ├── litus_longseg_ct2_3/         
    ├── litus_shortseg_ct2_1/
    ├── litus_shortseg_ct2_2/
    ├── litus_shortseg_ct2_3/        
    ├── multi_whisp_l_longseg_1/      # Fine-tuned PyTorch/HF weights   
    ├── multi_whisp_l_longseg_ct2_1/     
    ├── multi_whisp_l_longseg_ct2_2/
    ├── multi_whisp_l_longseg_ct2_3/         
    ├── multi_whisp_l_shortseg_ct2_1/
    ├── multi_whisp_l_shortseg_ct2_2/
    ├── multi_whisp_l_shortseg_ct2_3/
    ├── multi_whisp_m_longseg_1/      # Fine-tuned PyTorch/HF weights   
    ├── multi_whisp_m_longseg_ct2_1/     
    ├── multi_whisp_m_longseg_ct2_2/
    ├── multi_whisp_m_longseg_ct2_3/         
    ├── multi_whisp_m_shortseg_ct2_1/
    ├── multi_whisp_m_shortseg_ct2_2/
    ├── multi_whisp_m_shortseg_ct2_3/
    ├── multi_whisp_s_longseg_1/      # Fine-tuned PyTorch/HF weights   
    ├── multi_whisp_s_longseg_ct2_1/     
    ├── multi_whisp_s_longseg_ct2_2/
    ├── multi_whisp_s_longseg_ct2_3/         
    ├── multi_whisp_s_shortseg_ct2_1/
    ├── multi_whisp_s_shortseg_ct2_2/
    ├── multi_whisp_s_shortseg_ct2_3/         
    ├── sandiago_longseg_ct2_1/         
    ├── sandiago_longseg_ct2_2/
    ├── sandiago_longseg_ct2_2/
    ├── sandiago_shortseg_ct2_1/
    ├── sandiago_shortseg_ct2_2/
    └── sandiago_shortseg_ct2_3/

```

In the terminal, I need to convert the Hugging Face transformers weights for final_models into ones that CTranslate2 can use. For example, for the Sandiago model:

ct2-transformers-converter \
  --model /home/ialiani/thesis_Aliani/final_models/multi_whisp_s_longseg_3 \
  --output_dir /home/ialiani/thesis_Aliani/final_models/multi_whisp_s_longseg_ct2_3 \
  --quantization float16 \
  --copy_files tokenizer.json preprocessor_config.json \
  --force

ct2-transformers-converter \
  --model /home/ialiani/thesis_Aliani/final_models/sandiago_longseg_3 \
  --output_dir /home/ialiani/thesis_Aliani/final_models/sandiago_longseg_ct2_3 \
  --quantization float16 \
  --copy_files tokenizer.json preprocessor_config.json \
  --force

ct2-transformers-converter \
  --model /home/ialiani/thesis_Aliani/final_models/sandiago_shortseg_3 \
  --output_dir /home/ialiani/thesis_Aliani/final_models/sandiago_shortseg_ct2_3 \
  --quantization float16 \
  --copy_files tokenizer.json preprocessor_config.json \
  --force

### 5.1 Multilingual Whisper Large

In [30]:
multi_whisp_l_model_shortseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/multi_whisp_l_shortseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=multi_whisp_l_model_shortseg, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=MULTI_WHISP_L_SHORTSEG_RGN)

Found 875 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 875/875 [06:32<00:00,  2.23file/s]


Transcription complete!


In [31]:
multi_whisp_l_model_longseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/multi_whisp_l_longseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=multi_whisp_l_model_longseg, test_partition_dir=TEST_DIR_LONGSEG, output_dir=MULTI_WHISP_L_LONGSEG_RGN)

Found 206 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 206/206 [02:42<00:00,  1.27file/s]


Transcription complete!


In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del multi_whisp_l_model_shortseg
del multi_whisp_l_model_longseg


gc.collect()


if is_cuda:
    torch.cuda.empty_cache()

### 5.2 Whisper Medium

In [35]:
multi_whisp_m_model_shortseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/multi_whisp_m_shortseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=multi_whisp_m_model_shortseg, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=MULTI_WHISP_M_SHORTSEG_RGN)

Found 875 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 875/875 [05:02<00:00,  2.89file/s]


Transcription complete!


In [33]:
multi_whisp_m_model_longseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/multi_whisp_m_longseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=multi_whisp_m_model_longseg, test_partition_dir=TEST_DIR_LONGSEG, output_dir=MULTI_WHISP_M_LONGSEG_RGN)

Found 206 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 206/206 [02:06<00:00,  1.63file/s]


Transcription complete!


In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del multi_whisp_m_model_shortseg
del multi_whisp_m_model_longseg

gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

### 5.3 Whisper Small

In [37]:
multi_whisp_s_model_shortseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/multi_whisp_s_shortseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=multi_whisp_s_model_shortseg, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=MULTI_WHISP_S_SHORTSEG_RGN)

Found 875 .wav files to transcribe.


Transcribing audio files:   0%|          | 0/875 [00:00<?, ?file/s]

Transcribing audio files: 100%|██████████| 875/875 [04:05<00:00,  3.56file/s]


Transcription complete!


In [38]:
multi_whisp_s_model_longseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/multi_whisp_s_longseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=multi_whisp_s_model_longseg, test_partition_dir=TEST_DIR_LONGSEG, output_dir=MULTI_WHISP_S_LONGSEG_RGN)

Found 206 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 206/206 [01:19<00:00,  2.60file/s]


Transcription complete!


In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del multi_whisp_s_model_shortseg
del multi_whisp_s_model_longseg

gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

### 5.4 BuzzASR/Italian

In [40]:
buzz_model_shortseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/buzz_shortseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=buzz_model_shortseg, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=ITA_BUZZ_SHORTSEG_RGN)

Found 875 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 875/875 [46:20<00:00,  3.18s/file]  


Transcription complete!


In [ ]:
# buzz_model_longseg = load_asr_model(
#     model_path=Path("/home/ialiani/thesis_Aliani/final_models/buzz_longseg_ct2"),
#     device=device,
#     compute_type=compute_type
# )
# transcribe_folder(model=buzz_model_longseg, test_partition_dir=TEST_DIR_LONGSEG, output_dir=ITA_BUZZ_LONGSEG_RGN)

Found 206 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 206/206 [31:31<00:00,  9.18s/file]


Transcription complete!


In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del buzz_model_shortseg
# del buzz_model_longseg

gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

### 5.5 Litus model

In [42]:
litus_model_shortseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/litus_shortseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=litus_model_shortseg, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=ITA_LITUS_SHORTSEG_RGN)

Found 875 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 875/875 [03:54<00:00,  3.73file/s]


Transcription complete!


In [43]:
litus_model_longseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/litus_longseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=litus_model_longseg, test_partition_dir=TEST_DIR_LONGSEG, output_dir=ITA_LITUS_LONGSEG_RGN)

Found 206 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 206/206 [01:18<00:00,  2.62file/s]


Transcription complete!


In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del litus_model_shortseg
del litus_model_longseg

gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

### 5.6 Sandiago model

In [45]:
sandiago_model_shortseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/sandiago_shortseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=sandiago_model_shortseg, test_partition_dir=TEST_DIR_SHORTSEG, output_dir=ITA_SANDIAGO_SHORTSEG_RGN)

Found 875 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 875/875 [05:37<00:00,  2.59file/s]


Transcription complete!


In [46]:
sandiago_model_longseg = load_asr_model(
    model_path=Path("/home/ialiani/thesis_Aliani/final_models/sandiago_longseg_ct2_3"),
    device=device,
    compute_type=compute_type
)
transcribe_folder(model=sandiago_model_longseg, test_partition_dir=TEST_DIR_LONGSEG, output_dir=ITA_SANDIAGO_LONGSEG_RGN)

Found 206 .wav files to transcribe.


Transcribing audio files: 100%|██████████| 206/206 [02:40<00:00,  1.28file/s]


Transcription complete!


In [ ]:
# Delete model variable, force Python garbage collector to free CPU RAM memory, free GPU memory if running on CUDA
del sandiago_model_shortseg
del sandiago_model_longseg

gc.collect()

if is_cuda:
    torch.cuda.empty_cache()

## 6. ASR Evaluation

**Structure of the predictions folder.**

N.B. The folder "fine-tuned_results_1" is followed by "fine-tuned_results_2" and "fine-tuned_results_3" (they are not shown in the structure below because their content is identical to the "fine-tuned_results_1").

```text
thesis_Aliani/
└── predictions/
    |
    ├── baseline_results/          # Out-of-the-box predictions (in txt) on test set generated using baseline models   
    │   ├── baseline_longseg/
    |   |   ├── ita_buzz/
    |   |   ├── ita_litus/
    |   |   ├── ita_sandiago/
    |   |   ├── multi_whisper_l/
    |   |   ├── multi_whisper_m/
    |   |   └── multi_whisper_s/
    │   └── baseline_shortseg
    |       ├── ita_buzz/
    |       ├── ita_litus/
    |       ├── ita_sandiago/
    |       ├── multi_whisper_l/
    |       ├── multi_whisper_m/
    |       └── multi_whisper_s/
    |
    └── fine-tuned_results_1/        # Predictions (in txt) on test set generated using models fine-tuned on Romagnol data
        ├── finetuned_longseg/
        |   ├── ita_buzz/
        |   ├── ita_litus/
        |   ├── ita_sandiago/
        |   ├── multi_whisper_l/
        |   ├── multi_whisper_m/
        |   └── multi_whisper_s/
        └── finetuned_shortseg
            ├── ita_buzz/
            ├── ita_litus/
            ├── ita_sandiago/
            ├── multi_whisper_l/
            ├── multi_whisper_m/
            └── multi_whisper_s/
```

N.B. The model fine-tuned on shortseg was evaluated on the shortseg dataset, the one fine-tuned on longseg was evaluated on the longseg dataset. The two datasets differ for the average duration of each file).

In [48]:
# Run 1: Shortseg model
evaluate_and_append_html(gt_dir=TEST_DIR_SHORTSEG, base_dir=MULTI_WHISP_L_SHORTSEG, ft_dir=MULTI_WHISP_L_SHORTSEG_RGN, model_name="Whisper-Large-v3",
    partition="shortseg", output_html="asr_results_test_set_3.html")

# Run 2: Longseg model (appends next row to the same file)
evaluate_and_append_html(gt_dir=TEST_DIR_LONGSEG, base_dir=MULTI_WHISP_L_LONGSEG, ft_dir=MULTI_WHISP_L_LONGSEG_RGN, model_name="Whisper-Large-v3",
    partition="longseg", output_html="asr_results_test_set_3.html")

[Whisper-Large-v3 | shortseg]
  WER -> Base: 109.88% | FT: 62.1% (Δ: -47.78%)
  CER -> Base: 81.88% | FT: 20.36% (Δ: -61.52%)
[Whisper-Large-v3 | longseg]
  WER -> Base: 108.99% | FT: 59.35% (Δ: -49.64%)
  CER -> Base: 80.93% | FT: 18.77% (Δ: -62.16%)


In [49]:
# Run 1: Shortseg model
evaluate_and_append_html(gt_dir=TEST_DIR_SHORTSEG, base_dir=MULTI_WHISP_M_SHORTSEG, ft_dir=MULTI_WHISP_M_SHORTSEG_RGN, model_name="Whisper-Medium",
    partition="shortseg", output_html="asr_results_test_set_3.html")

# Run 2: Longseg model (appends next row to the same file)
evaluate_and_append_html(gt_dir=TEST_DIR_LONGSEG, base_dir=MULTI_WHISP_M_LONGSEG, ft_dir=MULTI_WHISP_M_LONGSEG_RGN, model_name="Whisper-Medium",
    partition="longseg", output_html="asr_results_test_set_3.html")

[Whisper-Medium | shortseg]
  WER -> Base: 86.03% | FT: 60.02% (Δ: -26.01%)
  CER -> Base: 41.35% | FT: 20.38% (Δ: -20.97%)
[Whisper-Medium | longseg]
  WER -> Base: 80.43% | FT: 60.04% (Δ: -20.39%)
  CER -> Base: 35.74% | FT: 20.17% (Δ: -15.57%)


In [50]:
# Run 1: Shortseg model
evaluate_and_append_html(gt_dir=TEST_DIR_SHORTSEG, base_dir=MULTI_WHISP_S_SHORTSEG, ft_dir=MULTI_WHISP_S_SHORTSEG_RGN, model_name="Whisper-Small",
    partition="shortseg", output_html="asr_results_test_set_3.html")

# Run 2: Longseg model (appends next row to the same file)
evaluate_and_append_html(gt_dir=TEST_DIR_LONGSEG, base_dir=MULTI_WHISP_S_LONGSEG, ft_dir=MULTI_WHISP_S_LONGSEG_RGN, model_name="Whisper-Small",
    partition="longseg", output_html="asr_results_test_set_3.html")

[Whisper-Small | shortseg]
  WER -> Base: 85.45% | FT: 62.61% (Δ: -22.84%)
  CER -> Base: 39.2% | FT: 22.41% (Δ: -16.79%)
[Whisper-Small | longseg]
  WER -> Base: 81.46% | FT: 58.82% (Δ: -22.64%)
  CER -> Base: 34.07% | FT: 19.68% (Δ: -14.39%)


In [52]:
# Run 1: Shortseg model
evaluate_and_append_html(gt_dir=TEST_DIR_SHORTSEG, base_dir=ITA_BUZZ_SHORTSEG, ft_dir=ITA_BUZZ_SHORTSEG_RGN, model_name="BuzzASR/italian",
    partition="shortseg", output_html="asr_results_test_set_3.html")

# Run 2: Longseg model (appends next row to the same file)
# evaluate_and_append_html(gt_dir=TEST_DIR_LONGSEG, base_dir=ITA_BUZZ_LONGSEG, ft_dir=ITA_BUZZ_LONGSEG_RGN, model_name="BuzzASR/italian",
#    partition="longseg", output_html="asr_results_test_set_3.html")

[BuzzASR/italian | shortseg]
  WER -> Base: 96.79% | FT: 89.39% (Δ: -7.40%)
  CER -> Base: 54.25% | FT: 50.47% (Δ: -3.78%)


In [53]:
# Run 1: Shortseg model
evaluate_and_append_html(gt_dir=TEST_DIR_SHORTSEG, base_dir=ITA_LITUS_SHORTSEG, ft_dir=ITA_LITUS_SHORTSEG_RGN, model_name="litus-ai/whisper-small-ita",
    partition="shortseg", output_html="asr_results_test_set_3.html")

# Run 2: Longseg model (appends next row to the same file)
evaluate_and_append_html(gt_dir=TEST_DIR_LONGSEG, base_dir=ITA_LITUS_LONGSEG, ft_dir=ITA_LITUS_LONGSEG_RGN, model_name="litus-ai/whisper-small-ita",
    partition="longseg", output_html="asr_results_test_set_3.html")

[litus-ai/whisper-small-ita | shortseg]
  WER -> Base: 87.84% | FT: 58.97% (Δ: -28.87%)
  CER -> Base: 53.09% | FT: 20.27% (Δ: -32.82%)
[litus-ai/whisper-small-ita | longseg]
  WER -> Base: 89.9% | FT: 59.29% (Δ: -30.61%)
  CER -> Base: 68.06% | FT: 20.64% (Δ: -47.42%)


In [54]:
# Run 1: Shortseg model
evaluate_and_append_html(gt_dir=TEST_DIR_SHORTSEG, base_dir=ITA_SANDIAGO_SHORTSEG, ft_dir=ITA_SANDIAGO_SHORTSEG_RGN, model_name="Sandiago21/whisper-large-v2-italian",
    partition="shortseg", output_html="asr_results_test_set_3.html")

# Run 2: Longseg model (appends next row to the same file)
evaluate_and_append_html(gt_dir=TEST_DIR_LONGSEG, base_dir=ITA_SANDIAGO_LONGSEG, ft_dir=ITA_SANDIAGO_LONGSEG_RGN, model_name="Sandiago21/whisper-large-v2-italian",
    partition="longseg", output_html="asr_results_test_set_3.html")

[Sandiago21/whisper-large-v2-italian | shortseg]
  WER -> Base: 86.26% | FT: 62.39% (Δ: -23.87%)
  CER -> Base: 39.74% | FT: 21.12% (Δ: -18.62%)
[Sandiago21/whisper-large-v2-italian | longseg]
  WER -> Base: 85.6% | FT: 61.55% (Δ: -24.05%)
  CER -> Base: 45.84% | FT: 20.49% (Δ: -25.35%)


In [ ]:
# List 3 run CSV files with ASR results on three folders of fine-tuned predictions.
run_csvs = [
    "asr_results_test_set_1.csv",
    "asr_results_test_set_2.csv",
    "asr_results_test_set_3.csv",
]

# Process and compute mean ± std across all runs
aggregated_df = aggregate_runs_and_export(
    csv_files=run_csvs, output_prefix="asr_aggregated_3runs"
)

# Display in notebook
display(aggregated_df)

Aggregated results generated successfully:
  - CSV:   /home/ialiani/thesis_Aliani/asr_aggregated_3runs.csv
  - HTML:  /home/ialiani/thesis_Aliani/asr_aggregated_3runs.html
  - LaTeX: /home/ialiani/thesis_Aliani/asr_aggregated_3runs.tex


,Model,Partition,Stage,Runs,Samples,WER (%),CER (%),Δ WER (%),Δ CER (%)
0,Whisper-Large-v3,shortseg,Baseline,3,875,109.88 ± 0.00,81.88 ± 0.00,-,-
1,Whisper-Large-v3,shortseg,Fine-Tuned,3,875,60.32 ± 1.82,19.79 ± 0.52,-49.56 ± 1.82,-62.09 ± 0.52
2,Whisper-Large-v3,longseg,Baseline,3,206,108.99 ± 0.00,80.93 ± 0.00,-,-
3,Whisper-Large-v3,longseg,Fine-Tuned,3,206,59.48 ± 1.49,18.78 ± 0.54,-49.51 ± 1.49,-62.15 ± 0.54
4,Whisper-Medium,shortseg,Baseline,3,875,86.03 ± 0.00,41.35 ± 0.00,-,-
5,Whisper-Medium,shortseg,Fine-Tuned,3,875,60.04 ± 0.30,20.41 ± 0.12,-25.99 ± 0.30,-20.94 ± 0.12
6,Whisper-Medium,longseg,Baseline,3,206,80.43 ± 0.00,35.74 ± 0.00,-,-
7,Whisper-Medium,longseg,Fine-Tuned,3,206,60.01 ± 0.34,20.15 ± 0.03,-20.42 ± 0.34,-15.59 ± 0.03
8,Whisper-Small,shortseg,Baseline,3,875,85.45 ± 0.00,39.20 ± 0.00,-,-
9,Whisper-Small,shortseg,Fine-Tuned,3,875,61.96 ± 0.58,21.96 ± 0.42,-23.49 ± 0.58,-17.24 ± 0.42


In [ ]:
# Run this cell to calculate WER and CER on single instances.

ref = "pr e' mi canèl e e' borgh che ènca se ò vést e' mond sol quest l'è stè e' mi mond clèt mond e' sarà ólta tot i cunfàin de' gnìnt tra e' vàird e e' turchìn in du che tot du insém e' mi ba e la mi ma i m'ciaparà par mèna e i m'dmandarà cume l'è andèda ad là"
hyp_1 = "e''prem i canel e' e' e' bòrgh che anca s'òvvvovvavvavvanvaa"

# Standardize text (lowercase + remove punctuation)
norm = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip()
])

ref_clean = norm(ref)
hyp_1_clean = norm(hyp_1)

print(f"CER: {jiwer.wer(ref_clean, hyp_1_clean) * 100:.2f}%")
print(f"CER: {jiwer.cer(ref_clean, hyp_1_clean) * 100:.2f}%")

CER: 91.94%
CER: 85.54%
